# Tahap 1 - Membangun Case Base
## Scraping & Preprocessing Putusan Wanprestasi - Pengadilan Negeri Surabaya

**Sumber:** Direktori Putusan Mahkamah Agung RI  
**Domain:** Perdata Gugatan Wanprestasi | **Pengadilan:** PN Surabaya | **Tahun:** 2025 | **Target:** 54 dokumen

---

## Persiapan Sebelum Menjalankan Notebook

### Langkah 1 - Buka Chrome dalam Mode Remote Debugging

Scraper ini menggunakan Selenium yang terhubung ke Chrome yang sudah terbuka (bukan membuka Chrome baru). Cara ini diperlukan agar verifikasi Cloudflare bisa diselesaikan secara manual oleh pengguna sebelum scraping dimulai.

Buka **CMD** (Command Prompt, bukan PowerShell), lalu jalankan perintah berikut:

```cmd
"C:\Program Files\Google\Chrome\Application\chrome.exe" --remote-debugging-port=9222 --user-data-dir="C:\chrome_debug"
```

**Penjelasan parameter:**
| Parameter | Keterangan |
|---|---|
| `--remote-debugging-port=9222` | Port yang digunakan Selenium untuk terhubung ke Chrome. Angka 9222 adalah default — **jangan diganti** kecuali port tersebut sudah dipakai proses lain |
| `--user-data-dir="C:\chrome_debug"` | Folder profil Chrome khusus untuk sesi debug ini. Dipisah dari profil Chrome utama agar tidak mengganggu sesi browser biasa |

> **Catatan:** Pastikan path `chrome.exe` sesuai dengan instalasi Chrome di komputer kamu. Jika Chrome diinstall di lokasi lain, sesuaikan path-nya, misalnya `"C:\Program Files (x86)\Google\Chrome\Application\chrome.exe"`.

---

### Langkah 2 - Buka Website Mahkamah Agung & Selesaikan Verifikasi

Setelah Chrome terbuka dalam mode debug:

1. Buka URL berikut di Chrome yang baru saja dibuka:
   ```
   https://putusan3.mahkamahagung.go.id
   ```
2. Akan muncul halaman verifikasi **Cloudflare** ("Checking your browser before accessing..."). Tunggu hingga selesai otomatis, atau klik checkbox "I am human" jika diminta.
3. Setelah berhasil masuk ke halaman utama Direktori Putusan MA RI, **jangan tutup browser tersebut**.
4. Lanjutkan ke pencarian putusan:
   - Klik menu **Pencarian**
   - Pilih **Jenis Perkara**: Perdata Gugatan
   - Pilih **Pengadilan**: Pengadilan Negeri Surabaya
   - Ketik keyword: **wanprestasi**
   - Klik **Cari**
   - Pastikan hasil pencarian sudah muncul di browser
5. Baru setelah itu jalankan Cell 3 ke bawah di notebook ini — Selenium akan mengambil alih browser yang sudah terbuka melalui port 9222.

> **Mengapa harus begini?** Website MA RI menggunakan proteksi Cloudflare yang memblokir akses otomatis (bot). Dengan membuka Chrome secara manual terlebih dahulu dan menyelesaikan verifikasi, Selenium tinggal menempel ke sesi yang sudah terverifikasi sehingga tidak terblokir.

---

### Langkah 3 - Jalankan Cell Secara Berurutan

| Cell | Fungsi | Keterangan |
|---|---|---|
| **Setup** | Jangkar working directory ke root repo | Jalankan **pertama kali selalu** |
| **Cell 1** | Install library | Cukup sekali |
| **Cell 2** | Konfigurasi & buat folder output | Setiap sesi |
| **Cell 3** | Fungsi koneksi, delay, HTTP request | Setiap sesi |
| **Cell 4** | Fungsi parse link, metadata, PDF ke TXT | Setiap sesi |
| **Cell 5** | Kumpulkan ~199 link unik dari MA RI | Cukup sekali |
| **Cell 5b** | *(Opsional)* Load link tersimpan untuk resume | Jika resume |
| **Cell 6** | Kosongkan folder PDF lama (fresh download) | Opsional |
| **Cell 7** | **Download PDF** dari MA RI (54 dokumen) | Proses utama |
| **Cell 8** | **Konversi PDF → TXT** (setelah semua PDF terkumpul) | Setelah Cell 7 |
| **Cell 9** | **Preprocessing** teks (cleaning, normalisasi, tokenisasi) | Setelah Cell 8 |
| **Cell 10** | Simpan log & metadata CSV | Setelah Cell 7 |
| **Cell 11** | Validasi hasil | Cek akhir |
| **Cell 12** | Ringkasan akhir | Cek akhir |

---

### Struktur Output
```
data/
├── raw/
│   ├── pdf/               ← PDF asli hasil unduhan (Cell 7)
│   └── txt/               ← TXT hasil konversi + preprocessing (Cell 8 & 9)
├── processed/
│   ├── tokens/            ← token JSON per dokumen (Cell 9)
│   └── metadata_raw.csv   ← metadata semua dokumen (Cell 10)
└── eval/
logs/
├── cleaning.log
├── links_putusan.json
├── preprocessing_summary.csv
└── validasi.csv
```


## Setup - Jangkar Working Directory ke Root Repository

**Jalankan cell ini SELALU sebagai cell pertama**, sebelum cell lain di notebook ini.

Notebook ini disimpan di dalam folder `notebooks/`, tapi seluruh path data di notebook ini
(mis. `'data/processed/cases.csv'`) ditulis **relatif terhadap root repository**, bukan terhadap
folder `notebooks/`. Secara default, Jupyter menjalankan kernel dengan *working directory* = folder
tempat file `.ipynb` itu sendiri berada. Tanpa cell ini, path seperti `'data/raw'` akan dicari di
`notebooks/data/raw` (tidak ada) dan menyebabkan `FileNotFoundError`.

Cell ini **aman dijalankan berkali-kali** (idempotent) — begitu working directory sudah berada di
root repository (folder yang punya subfolder `data/`), cell ini tidak akan berpindah lagi.

In [ ]:
import os

# Jika folder 'data/' tidak ada di working directory saat ini, TAPI ada satu
# level di atasnya -> kita sedang di dalam notebooks/, pindah ke root repo.
if not os.path.isdir('data') and os.path.isdir(os.path.join('..', 'data')):
    os.chdir('..')

print('Working directory aktif :', os.getcwd())
print('Isi folder saat ini     :', sorted(os.listdir('.')))

assert os.path.isdir('data'), (
    "Folder 'data/' tidak ditemukan dari working directory ini.\n"
    "Pastikan struktur repo: <root>/notebooks/xx.ipynb dengan <root>/data/ di sebelahnya,\n"
    "dan notebook dibuka dari dalam struktur repo tersebut (bukan dipindah sendirian)."
)


## Cell 1 — Install Library

In [2]:
import subprocess, sys

packages = [
    'selenium',
    'beautifulsoup4',
    'pandas',
    'tqdm',
    'pdfminer.six',
    'lxml',
    'requests',
]

for pkg in packages:
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', pkg, '-q'],
        capture_output=True, text=True
    )
    status = '✅' if result.returncode == 0 else '❌'
    print(f'{status} {pkg}')

print('\n✅ Semua library terinstall!')


✅ selenium
✅ beautifulsoup4
✅ pandas
✅ tqdm
✅ pdfminer.six
✅ lxml
✅ requests

✅ Semua library terinstall!


## Cell 2 — Konfigurasi & Inisialisasi Direktori

In [2]:
import os, re, time, random, json, math
import pandas as pd
from datetime import datetime
from pathlib import Path
from tqdm import tqdm
from urllib.parse import urljoin
from bs4 import BeautifulSoup
from pdfminer.high_level import extract_text as pdf_extract

# ─── KONFIGURASI ───────────────────────────────────────────────────────────
CONFIG = {
    # Target pengadilan
    'pengadilan_kode' : 'pn-surabaya',
    'pengadilan_nama' : 'Pengadilan Negeri Surabaya',

    # Kategori di sistem MA RI
    'kategori' : 'wanprestasi-1',

    # Tahun putusan
    'tahunjenis' : 'putus',
    'tahun' : '2025',

    # Jumlah dokumen target sesuai instruksi tugas
    # Total kasus tersedia: 199 — kita ambil 50
    'target_dokumen'  : 80,

    # Human-like delay (detik) — distribusi Gaussian
    'delay_mean'      : 4.0,
    'delay_std'       : 1.5,
    'delay_min'       : 2.5,
    'delay_max'       : 9.0,

    # Jeda lebih panjang setelah N request
    'long_pause_every': 8,
    'long_pause_min'  : 15,
    'long_pause_max'  : 30,

    # Direktori output sesuai instruksi Tugas Tahap 1
    'dir_raw_pdf'     : 'data/raw/pdf',   # PDF asli hasil download
    'dir_raw_txt'     : 'data/raw',       # Tempat case_001.txt, dll. (hasil konversi dari PDF)
    'dir_logs'        : 'logs',           # Tempat cleaning.log
    'dir_processed'   : 'data/processed',
    'dir_eval'        : 'data/eval',

    # Validasi teks minimum (80% isi putusan tersedia)
    'min_text_chars'  : 800,
}

BASE_DOMAIN = 'https://putusan3.mahkamahagung.go.id'

# ─── URL builder ───────────────────────────────────────────────────────────
def url_halaman(page=1):
    return (
        f"{BASE_DOMAIN}/direktori/index/"
        f"pengadilan/{CONFIG['pengadilan_kode']}/"
        f"kategori/{CONFIG['kategori']}/"
        f"tahunjenis/{CONFIG['tahunjenis']}/"
        f"tahun/{CONFIG['tahun']}.html"
        if page == 1 else
        f"{BASE_DOMAIN}/direktori/index/"
        f"pengadilan/{CONFIG['pengadilan_kode']}/"
        f"kategori/{CONFIG['kategori']}/"
        f"tahunjenis/{CONFIG['tahunjenis']}/"
        f"tahun/{CONFIG['tahun']}/"
        f"page/{page}.html"
    )

# ─── Buat direktori output ─────────────────────────────────────────────────
for d in [CONFIG['dir_raw_pdf'], CONFIG['dir_raw_txt'],
          CONFIG['dir_logs'], CONFIG['dir_processed'], CONFIG['dir_eval']]:
    os.makedirs(d, exist_ok=True)

# ─── Print ringkasan ───────────────────────────────────────────────────────
print('=' * 62)
print(f"  Pengadilan  : {CONFIG['pengadilan_nama']}")
print(f"  Kategori    : {CONFIG['kategori']} (perdata wanprestasi)")
print(f"  Tahun       : {CONFIG['tahun']}")
print(f"  Target      : {CONFIG['target_dokumen']} dokumen (dari 199 tersedia)")
print(f"  URL Hal. 1  : {url_halaman(1)}")
print('=' * 62)
print()
print('📁 Struktur folder output:')
print('   data/raw/pdf/   ← PDF asli hasil download')
print('   data/raw/       ← TXT hasil konversi dari PDF (case_001.txt, ...)')
print('   data/processed/ ← metadata CSV')
print('   logs/           ← cleaning.log, validasi.csv')


  Pengadilan  : Pengadilan Negeri Surabaya
  Kategori    : wanprestasi-1 (perdata wanprestasi)
  Tahun       : 2025
  Target      : 80 dokumen (dari 199 tersedia)
  URL Hal. 1  : https://putusan3.mahkamahagung.go.id/direktori/index/pengadilan/pn-surabaya/kategori/wanprestasi-1/tahunjenis/putus/tahun/2025.html

📁 Struktur folder output:
   data/raw/pdf/   ← PDF asli hasil download
   data/raw/       ← TXT hasil konversi dari PDF (case_001.txt, ...)
   data/processed/ ← metadata CSV
   logs/           ← cleaning.log, validasi.csv


## Cell 3 — Fungsi Koneksi, Delay & HTTP Request

In [3]:
import re, time, random
from urllib.parse import urljoin
from bs4 import BeautifulSoup
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

# ─── LOGIKA JEDA MANUSIA (HUMAN-LIKE TIMING) ────────────────────────────────
def human_delay(mean=None, std=None):
    """Jeda dinamis dengan distribusi Gaussian agar tidak terdeteksi bot."""
    mean = mean or CONFIG['delay_mean']
    std  = std  or CONFIG['delay_std']
    t = random.gauss(mean, std)
    t = max(CONFIG['delay_min'], min(CONFIG['delay_max'], t))
    time.sleep(t)

def long_pause():
    """Jeda panjang untuk menyimulasikan manusia sedang membaca halaman."""
    t = random.uniform(CONFIG['long_pause_min'], CONFIG['long_pause_max'])
    print(f'   ☕ Server Proteksi: Mengambil jeda istirahat {t:.1f} detik...')
    time.sleep(t)

# ─── KONEKSI DETEKSI CHROME ACTIVE ──────────────────────────────────────────
options = Options()
options.debugger_address = "127.0.0.1:9222"

try:
    driver = webdriver.Chrome(options=options)
    print(f'✅ KONEKSI AMAN: Terhubung ke jendela Chrome: "{driver.title}"')
except Exception as e:
    print(f'❌ KONEKSI GAGAL: Pastikan Chrome via CMD port 9222 sudah menyala! Error: {e}')

# ─── FUNGSI GET HALAMAN DENGAN DETEKSI CLOUDFLARE CERDAS ────────────────────
def get_halaman(url, retry=3):
    """
    Membuka URL menggunakan Chrome aktif. Jika tertahan Cloudflare,
    Python akan menghentikan proses sementara memberi waktu Anda mencentangnya,
    lalu otomatis melanjutkan setelah gerbang terbuka.
    """
    for attempt in range(retry):
        try:
            driver.get(url)
            time.sleep(random.uniform(2.5, 4.0))

            if "Just a moment" in driver.title or "Cloudflare" in driver.title or "Attention Required" in driver.title:
                print(f'   ⚠️  TERDETEKSI CLOUDFLARE pada tautan: {url}')
                print('   📢 TINDAKAN MANUAL: Silakan CENTANG kotak "Verify you are human" di browser Anda sekarang!')
                for detik in range(60):
                    time.sleep(1)
                    if "Just a moment" not in driver.title and "Cloudflare" not in driver.title:
                        print("   🎉 Terverifikasi! Gerbang terbuka, melanjutkan scraping...")
                        break
                else:
                    print("   ❌ Timeout: Anda terlalu lama mencentang Cloudflare, mencoba ulang request...")
                    continue

            if "Just a moment" not in driver.title:
                return driver.page_source

        except Exception as e:
            print(f'   ❌ Upaya {attempt+1}/{retry} gagal memuat halaman: {e}')

        time.sleep(random.uniform(3, 5))
    return None

print('✅ Fungsi get_halaman() dengan Fitur Tunggu Cloudflare SIAP!')


✅ KONEKSI AMAN: Terhubung ke jendela Chrome: "Direktori Putusan"
✅ Fungsi get_halaman() dengan Fitur Tunggu Cloudflare SIAP!


## Cell 4 — Fungsi Parse Link, Metadata & PDF ke TXT

> Filter otomatis: hanya mengambil link putusan dari halaman hasil pencarian PN Surabaya, bukan dari sidebar pengadilan lain.

In [4]:
from bs4 import BeautifulSoup

with open('logs/debug_page1.html', 'r', encoding='utf-8') as f:
    html = f.read()

soup = BeautifulSoup(html, 'lxml')

# Cek apakah ada <table>
tables = soup.find_all('table')
print(f"Jumlah <table>: {len(tables)}")

# Cek link putusan dari SEMUA elemen (pakai regex yang lama)
import re
pola = re.compile(r'/direktori/putusan/[a-zA-Z0-9]+\.html')
semua_a = soup.find_all('a', href=True)
link_putusan = [a['href'] for a in semua_a if pola.search(a['href'])]
print(f"Total link putusan ditemukan (tanpa filter): {len(link_putusan)}")
print("Contoh 3 link:", link_putusan[:3])

# Cek parent dari link pertama
if link_putusan:
    a_el = soup.find('a', href=re.compile(r'/direktori/putusan/'))
    print(f"\nParent tag: {a_el.find_parent().name}")
    print(f"Parent class: {a_el.find_parent().get('class')}")
    print(f"Grandparent tag: {a_el.find_parent().find_parent().name}")
    print(f"Grandparent class: {a_el.find_parent().find_parent().get('class')}")

Jumlah <table>: 1
Total link putusan ditemukan (tanpa filter): 43
Contoh 3 link: ['https://putusan3.mahkamahagung.go.id/direktori/putusan/zaf0e7fc74efbb048336323335393532.html', 'https://putusan3.mahkamahagung.go.id/direktori/putusan/zaf0e7fbcdf703fcb3a4323335353132.html', 'https://putusan3.mahkamahagung.go.id/direktori/putusan/zaf0e7fbcdf6662cb3a4323335353132.html']

Parent tag: strong
Parent class: None
Grandparent tag: div
Grandparent class: ['entry-c']


In [6]:
import re, time, random
from bs4 import BeautifulSoup
from urllib.parse import urljoin

# ─── PARSER INDEKS UTAMA (MENGAMBIL LINK DETAIL PUTUSAN) ───────────────────
def parse_link_putusan(html_source):
    """
    [FIX v3] Hanya ambil link dari div.entry-c (tabel hasil pencarian utama).
    div.entry-title = sidebar pengadilan lain → diabaikan.
    """
    soup = BeautifulSoup(html_source, 'lxml')
    links = []
    seen  = set()

    pola_putusan = re.compile(r'/direktori/putusan/[a-zA-Z0-9]+\.html')

    for a in soup.find_all('a', href=True):
        href = a['href']
        if not pola_putusan.search(href):
            continue
        if href in seen:
            continue

        # Cek grandparent class — hanya ambil dari entry-c
        parent = a.find_parent()
        grandparent = parent.find_parent() if parent else None
        gp_class = grandparent.get('class', []) if grandparent else []

        if 'entry-c' in gp_class:
            full_url = urljoin(BASE_DOMAIN, href)
            links.append(full_url)
            seen.add(href)

    return links


# ─── PARSER METADATA HALAMAN DETAIL ────────────────────────────────────────
def parse_metadata(html_source, url_detail):
    """
    Mengekstrak tabel informasi hukum (metadata) dari halaman detail putusan.
    """
    soup = BeautifulSoup(html_source, 'lxml')

    def cari(label_kw):
        """Mencari teks isi (value) berdasarkan kata kunci label di baris tabel."""
        for row in soup.find_all('tr'):
            teks_row = row.get_text(separator=' ', strip=True)
            for kw in label_kw:
                if kw.lower() in teks_row.lower():
                    tds = row.find_all('td')
                    if len(tds) >= 2:
                        return tds[-1].get_text(strip=True)
        return 'Tidak Tertera'

    amar = 'Tidak Tertera'
    for kw_amar in ['mengadili', 'amar', 'menyatakan', 'menghukum']:
        pattern = re.compile(kw_amar, re.IGNORECASE)
        elemen  = soup.find(string=pattern)
        if elemen:
            parent = elemen.find_parent()
            if parent:
                amar = parent.get_text(strip=True)[:300]
                break

    return {
        'no_perkara'   : cari(['Nomor', 'No. Perkara', 'Nomor Perkara']),
        'tanggal'      : cari(['Tanggal', 'Tgl', 'Tanggal Putusan']),
        'jenis_perkara': cari(['Jenis', 'Klasifikasi', 'Jenis Perkara']),
        'pengadilan'   : cari(['Pengadilan', 'Lembaga']),
        'pihak'        : cari(['Para Pihak', 'Pihak', 'Penggugat', 'Tergugat']),
        'pasal'        : cari(['Pasal', 'Dasar Hukum']),
        'amar_putusan' : amar,
        'url_sumber'   : url_detail,
    }

print('✅ Fungsi parser link dan ekstraksi metadata SIAP!')
print()
# Debug opsional: cek halaman 1
html_test = get_halaman(url_halaman(1))
if html_test:
    print(f'✅ HTML didapat: {len(html_test)} karakter')
    os.makedirs('logs', exist_ok=True)
    with open('logs/debug_page1.html', 'w', encoding='utf-8') as f:
        f.write(html_test)
    print('💾 Disimpan ke logs/debug_page1.html')
    print(f'Ada kata "putusan": {"putusan" in html_test.lower()}')
    print(f'Ada tag <a href: {html_test.count("<a href")}')
else:
    print('❌ HTML None — halaman gagal diambil')


✅ Fungsi parser link dan ekstraksi metadata SIAP!

✅ HTML didapat: 114314 karakter
💾 Disimpan ke logs/debug_page1.html
Ada kata "putusan": True
Ada tag <a href: 214


In [7]:
import re, time, random
from bs4 import BeautifulSoup
from urllib.parse import urljoin

# ─── [PERBAIKAN KETAT] PARSER INDEKS UTAMA (STERILISASI LINK) ───────────────────
def parse_link_putusan(html_source):
    """
    [FIX v4] Mengambil link detail dari div.entry-c, 
    dan LANGSUNG menyaring agar terbebas dari Akta Perdamaian/Penetapan/Pdt.P.
    """
    soup = BeautifulSoup(html_source, 'lxml')
    links = []
    seen  = set()

    pola_putusan = re.compile(r'/direktori/putusan/[a-zA-Z0-9]+\.html')

    # Cari seluruh tag anchor (a) yang memiliki href putusan
    for a in soup.find_all('a', href=True):
        href = a['href']
        
        # Validasi format URL putusan murni
        if not pola_putusan.search(href) or '/download/' in href or '/berkas/' in href:
            continue
        if href in seen:
            continue

        # Cek hirarki pembungkus untuk memastikan itu ada di area hasil pencarian utama
        parent = a.find_parent()
        grandparent = parent.find_parent() if parent else None
        gp_class = grandparent.get('class', []) if grandparent else []

        # Di web MA v3, hasil pencarian utama dibungkus kelas 'entry-c'
        if 'entry-c' in gp_class:
            
            # ── SIMULASI FILTER CONTAINER SEBELUM LINK DILOLOSKAN ──
            # Ambil boks pembungkus terdekat untuk melihat teks informasinya
            parent_box = a.find_parent(['div', 'li', 'tr', 'td'])
            teks_kontainer = parent_box.get_text(" ", strip=True).upper() if parent_box else ""
            
            # Kata kunci yang mutlak dilarang masuk Case Base Putusan
            kata_larangan = ['AKTA PERDAMAIAN', 'PERDAMAIAN', 'VAN DADING', 'AKTA', 'PENETAPAN', 'PDT.P']
            
            # Jika terdeteksi kata larangan di boks daftar, skip link ini!
            if any(k in teks_kontainer for k in kata_larangan):
                continue
                
            full_url = urljoin(BASE_DOMAIN, href)
            links.append(full_url)
            seen.add(href)

    return links


# ─── PARSER METADATA HALAMAN DETAIL ────────────────────────────────────────
def parse_metadata(html_source, url_detail):
    """
    Mengekstrak tabel informasi hukum (metadata) dari halaman detail putusan.
    """
    soup = BeautifulSoup(html_source, 'lxml')

    def cari(label_kw):
        """Mencari teks isi (value) berdasarkan kata kunci label di baris tabel."""
        for row in soup.find_all('tr'):
            teks_row = row.get_text(separator=' ', strip=True)
            for kw in label_kw:
                if kw.lower() in teks_row.lower():
                    tds = row.find_all('td')
                    if len(tds) >= 2:
                        return tds[-1].get_text(strip=True)
        return 'Tidak Tertera'

    # Ekstraksi ringkasan amar putusan halaman web
    amar = 'Tidak Tertera'
    for kw_amar in ['mengadili', 'amar', 'menyatakan', 'menghukum']:
        pattern = re.compile(kw_amar, re.IGNORECASE)
        elemen  = soup.find(string=pattern)
        if elemen:
            parent = elemen.find_parent()
            if parent:
                # Bersihkan spasi ganda atau baris baru agar tidak merusak format CSV/JSON
                teks_bersih = ' '.join(parent.get_text(strip=True).split())
                amar = teks_bersih[:300]
                break

    return {
        'no_perkara'   : cari(['Nomor', 'No. Perkara', 'Nomor Perkara']),
        'tanggal'      : cari(['Tanggal', 'Tgl', 'Tanggal Putusan']),
        'jenis_perkara': cari(['Jenis', 'Klasifikasi', 'Jenis Perkara']),
        'pengadilan'   : cari(['Pengadilan', 'Lembaga']),
        'pihak'        : cari(['Para Pihak', 'Pihak', 'Penggugat', 'Tergugat']),
        'pasal'        : cari(['Pasal', 'Dasar Hukum']),
        'amar_putusan' : amar,
        'url_sumber'   : url_detail,
    }

print('✅ Fungsi parser link STERIL dan ekstraksi metadata SIAP!')
print()

# Debug opsional: cek halaman 1
html_test = get_halaman(url_halaman(1))
if html_test:
    print(f'✅ HTML didapat: {len(html_test)} karakter')
    os.makedirs('logs', exist_ok=True)
    with open('logs/debug_page1.html', 'w', encoding='utf-8') as f:
        f.write(html_test)
    print('💾 Disimpan ke logs/debug_page1.html')
    
    # Cek seberapa ampuh fungsi baru mendeteksi link murni
    test_links = parse_link_putusan(html_test)
    print(f'📊 Hasil Test Parser Baru pada Hal 1: Menemukan {len(test_links)} link lolos filter.')
else:
    print('❌ HTML None — halaman gagal diambil')

✅ Fungsi parser link STERIL dan ekstraksi metadata SIAP!

✅ HTML didapat: 114314 karakter
💾 Disimpan ke logs/debug_page1.html
📊 Hasil Test Parser Baru pada Hal 1: Menemukan 40 link lolos filter.


## Cell 5 — Kumpulkan Link Putusan (Pagination + Deduplikasi Hash)

> Mengumpulkan semua link dari halaman indeks MA RI (PN Surabaya, wanprestasi, 2025).
> Deduplikasi otomatis berbasis hash URL → ~199 link unik tanpa duplikat.

In [18]:
import json
import os
import time
import re
from bs4 import BeautifulSoup

print('🔍 Mulai mengumpulkan link putusan (DENGAN AUTO-SAVE & FILTER STABIL)...')
print(f'   Target download                  : {CONFIG["target_dokumen"]} dokumen')
print(f'   URL Hal. 1 : {url_halaman(1)}')
print()

semua_link  = []
hash_seen   = set()  
halaman     = 1
max_halaman = 25  
request_count = 0
gagal_berturut_turut = 0 

# Buat folder logs di awal agar aman
os.makedirs('logs', exist_ok=True)

while halaman <= max_halaman:
    url = url_halaman(halaman)
    print(f' 📄 Mengambil Link Halaman {halaman}/{max_halaman}: {url[-60:]}')

    html = get_halaman(url)
    if html is None:
        print(f'   ❌ Gagal mengambil halaman {halaman}. Skip ke halaman berikutnya...')
        halaman += 1
        time.sleep(3) 
        continue

    # ── PARSING DENGAN TARGET CONTAINER SPESIFIK ──
    soup = BeautifulSoup(html, 'html.parser')
    links_lolos_filter = []
    
    # Di web MA v3, setiap perkara biasanya dibungkus dalam tag <tr> (tabel) atau <li> / <div> bertipe khusus
    # Kita cari text langsung dari elemen yang memiliki link putusan murni
    for a_tag in soup.find_all('a', href=True):
        if '/putusan/' in a_tag['href'] and '/download/' not in a_tag['href'] and '/berkas/' not in a_tag['href']:
            url_putusan = a_tag['href']
            
            # Cari boks pembungkus terdekatnya saja (biar tidak melebar ke teks tetangga)
            parent_box = a_tag.find_parent(['td', 'div', 'li'])
            teks_kontainer = parent_box.get_text(" ", strip=True).upper() if parent_box else ""
            
            # Kata larangan yang mutlak dibuang
            kata_larangan = ['AKTA PERDAMAIAN', 'PERDAMAIAN', 'VAN DADING', 'AKTA', 'PENETAPAN', 'PDT.P']
            
            if any(k in teks_kontainer for k in kata_larangan):
                continue # Skip jika terbukti akta/penetapan
                
            if url_putusan not in links_lolos_filter:
                links_lolos_filter.append(url_putusan)

    # ── VALIDASI DATA KOSONG ──
    if len(links_lolos_filter) == 0:
        gagal_berturut_turut += 1
        print(f'   ⚠️  Halaman {halaman}: 0 link lolos filter.')
        
        if gagal_berturut_turut >= 3:
            print('   ℹ️ Sudah 3 halaman berturut-turut kosong. Menghentikan pencarian...')
            break
            
        halaman += 1
        continue
    else:
        gagal_berturut_turut = 0 

    # Deduplikasi berbasis hash URL
    ditambah = 0
    duplikat  = 0
    for lnk in links_lolos_filter:
        match = re.search(r'/direktori/putusan/([a-zA-Z0-9]+)\.html', lnk)
        if match:
            hash_url = match.group(1)
            if hash_url not in hash_seen:
                hash_seen.add(hash_url)
                semua_link.append(lnk)
                ditambah += 1
            else:
                duplikat += 1

    print(f'     → Lolos: {len(links_lolos_filter)} | Baru Unik: {ditambah} | Total Terkumpul: {len(semua_link)}')

    # ── [PERBAIKAN FITUR] AMANKAN DATA SECARA REAL-TIME KE JSON ──
    with open('logs/links_putusan.json', 'w', encoding='utf-8') as f:
        json.dump(semua_link, f, indent=2, ensure_ascii=False)

    halaman += 1
    request_count += 1
    
    # Jeda dinamis agar tidak ceket kelamaan
    time.sleep(2)

print()
print('=' * 60)
print(f'✅ PROSES PENGUMPULAN SELESAI!')
print(f'📊 Berhasil mengunci {len(semua_link)} link STERIL langsung di logs/links_putusan.json')
print('=' * 60)

🔍 Mulai mengumpulkan link putusan (DENGAN AUTO-SAVE & FILTER STABIL)...
   Target download                  : 80 dokumen
   URL Hal. 1 : https://putusan3.mahkamahagung.go.id/direktori/index/pengadilan/pn-surabaya/kategori/wanprestasi-1/tahunjenis/putus/tahun/2025.html

 📄 Mengambil Link Halaman 1/25: baya/kategori/wanprestasi-1/tahunjenis/putus/tahun/2025.html
     → Lolos: 42 | Baru Unik: 42 | Total Terkumpul: 42
 📄 Mengambil Link Halaman 2/25: tegori/wanprestasi-1/tahunjenis/putus/tahun/2025/page/2.html
     → Lolos: 41 | Baru Unik: 21 | Total Terkumpul: 63
 📄 Mengambil Link Halaman 3/25: tegori/wanprestasi-1/tahunjenis/putus/tahun/2025/page/3.html
     → Lolos: 38 | Baru Unik: 18 | Total Terkumpul: 81
 📄 Mengambil Link Halaman 4/25: tegori/wanprestasi-1/tahunjenis/putus/tahun/2025/page/4.html
     → Lolos: 41 | Baru Unik: 18 | Total Terkumpul: 99
 📄 Mengambil Link Halaman 5/25: tegori/wanprestasi-1/tahunjenis/putus/tahun/2025/page/5.html
     → Lolos: 43 | Baru Unik: 20 | Total Terk

### ♻️ Cell 5b — (Opsional) Load Link dari File Sebelumnya

> Jalankan **hanya jika** Cell 5 sudah pernah berjalan dan ingin melanjutkan tanpa scraping ulang.

In [19]:
# Muat ulang link yang sudah tersimpan (lewati jika Cell 5 baru saja dijalankan)
import json, os

link_file = 'logs/links_putusan.json'
if os.path.exists(link_file):
    with open(link_file, 'r', encoding='utf-8') as f:
        semua_link = json.load(f)
    print(f'✅ Dimuat dari {link_file}: {len(semua_link)} link siap diproses')
else:
    print('⚠️  File tidak ditemukan. Jalankan Cell 5 terlebih dahulu.')


✅ Dimuat dari logs/links_putusan.json: 208 link siap diproses


## Cell 6 — (Opsional) Kosongkan Folder PDF untuk Fresh Download

> Jalankan hanya jika ingin memulai download dari awal. **Lewati** jika ingin melanjutkan download sebelumnya.

In [11]:
import os
import glob

# Path folder PDF Anda
folder_pdf = CONFIG['dir_raw_pdf']

# Cari semua file PDF di dalam folder tersebut
files = glob.glob(os.path.join(folder_pdf, '*.pdf'))

# Hapus semua file PDF lama
for f in files:
    try:
        os.remove(f)
    except Exception as e:
        print(f"Gagal menghapus {f}: {e}")

print(f"🧹 Folder {folder_pdf} berhasil dikosongkan! Siap memulai unduhan bersih (fresh download).")

# [FIX v2] Bersihkan juga folder TXT agar tidak ada sisa run sebelumnya
# Tanpa ini, file TXT lama dari pengadilan lain bisa tetap ada walau PDF sudah bersih
import glob as _glob
txt_lama = _glob.glob(os.path.join(CONFIG['dir_raw_txt'], '*.txt'))
for f in txt_lama:
    try:
        os.remove(f)
    except Exception as e:
        print(f"Gagal hapus TXT {f}: {e}")
if txt_lama:
    print(f"🧹 {len(txt_lama)} file TXT lama di {CONFIG['dir_raw_txt']}/ juga dihapus (fresh start).")
else:
    print(f"ℹ️  Tidak ada file TXT lama di {CONFIG['dir_raw_txt']}/. Folder sudah bersih.")


🧹 Folder data/raw/pdf berhasil dikosongkan! Siap memulai unduhan bersih (fresh download).
ℹ️  Tidak ada file TXT lama di data/raw/. Folder sudah bersih.


## Cell 7 — Proses Utama: Download PDF dari MA RI

> Download **50 PDF** dari link yang sudah dikumpulkan di Cell 5.
> PDF disimpan ke `data/raw/pdf/`. Konversi ke TXT dilakukan di Cell 8 setelah semua PDF terkumpul.
> Berhenti otomatis setelah 50 PDF berhasil diunduh.

In [20]:
import os, time, re, glob, shutil
from datetime import datetime
from selenium.webdriver.common.by import By

DIR_PDF = os.path.abspath(CONFIG['dir_raw_pdf'])  # data/raw/pdf/
os.makedirs(DIR_PDF, exist_ok=True)

# ── Arahkan folder download Chrome ke DIR_PDF ──────────────────────────────
try:
    driver.execute_cdp_cmd("Page.setDownloadBehavior", {
        "behavior": "allow",
        "downloadPath": DIR_PDF
    })
    print(f'⚙️  Folder download Chrome diarahkan ke:\n    👉 {DIR_PDF}\n')
except Exception as e:
    print(f'⚠️  Gagal mengunci folder via DevTools ({e}), file mungkin ke Downloads default.\n')

# ── Variabel tracking ──────────────────────────────────────────────────────
TARGET_DOKUMEN    = CONFIG['target_dokumen']  # 50
dokumen_terproses = 0
gagal_list        = []
data_metadata     = []
log_lines         = []

log_lines.append(f'Mulai: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
log_lines.append(f'Target: {TARGET_DOKUMEN} dokumen dari {len(semua_link)} link tersedia')
log_lines.append('-' * 50)

print('=' * 62)
print('   PROSES UTAMA: PDF DOWNLOADER MURNI (FIXED WINDOWS WINERROR 183)')
print(f'   Total link: {len(semua_link)} | Target: {TARGET_DOKUMEN} dokumen')
print('=' * 62)
print()

for i, url_detail in enumerate(semua_link, start=1):
    if dokumen_terproses >= TARGET_DOKUMEN:
        print(f'🎯 TARGET TERPENUHI! {TARGET_DOKUMEN} PDF berhasil diunduh.')
        break

    # ── Bersihkan & standarisasi URL ──────────────────────────────────────
    url_bersih = url_detail.strip()
    if 'putusan3.mahkamahagung.go.id' not in url_bersih:
        if 'ung.go.id' in url_bersih:
            url_bersih = url_bersih.split('ung.go.id')[-1]
        elif 'mahagung.go.id' in url_bersih:
            url_bersih = url_bersih.split('mahagung.go.id')[-1]
        if not url_bersih.startswith('/'):
            url_bersih = '/' + url_bersih
        url_bersih = f"https://putusan3.mahkamahagung.go.id{url_bersih}"

    # Penentuan ID Kasus Berdasarkan Jumlah yang Sukses Masuk Folder
    case_id  = f'case_{dokumen_terproses + 1:03d}'
    path_pdf = os.path.join(DIR_PDF, f'{case_id}.pdf')

    print(f'📦 [{dokumen_terproses}/{TARGET_DOKUMEN}] Link {i}/{len(semua_link)}')
    print(f'   URL: {url_bersih}')

    try:
        # 1. Buka halaman detail dengan waktu tunggu muat halaman lebih longgar
        driver.get(url_bersih)
        time.sleep(7) # Ditambah dari 5 ke 7 detik agar HTML tombol tidak terlewat (Solusi Masalah 1)

        teks_visible = driver.find_element(By.TAG_NAME, 'body').text.upper()
        baris_awal = '\n'.join(teks_visible.split('\n')[:15])

        kata_skip = ['AKTA PERDAMAIAN', 'PENETAPAN']
        if any(k in baris_awal for k in kata_skip):
            print(f'   ⏭️  [SKIP] Bukan putusan: {baris_awal[:80]}')
            gagal_list.append({'url': url_bersih, 'alasan': 'Bukan putusan'})
            log_lines.append(f'SKIP [bukan-putusan] {url_bersih}')
            print('-' * 50)
            human_delay()
            continue


        # 2. [FIX v2] Validasi: pastikan halaman ini benar-benar dari PN Surabaya
        # Cek teks halaman sebelum proses apapun — cegah PDF pengadilan lain ikut masuk
        page_src_lower = driver.page_source.lower()
        kata_kunci_pn  = ['pengadilan negeri surabaya', 'pn surabaya', 'pn-surabaya']
        if not any(kw in page_src_lower for kw in kata_kunci_pn):
            print('   ⏭️  [SKIP-BUKAN-PN-SBY] Halaman ini bukan putusan PN Surabaya. Dilewati.')
            gagal_list.append({'url': url_bersih, 'alasan': 'Bukan PN Surabaya'})
            log_lines.append(f'SKIP [bukan-pn-sby] {url_bersih}')
            print('-' * 50)
            continue

        # 3. Ekstrak metadata dari halaman detail HTML
        meta = parse_metadata(driver.page_source, url_bersih)
        
        # ── FILTER KETAT 1: BLOKIR PENETAPAN & AKTA PERDAMAIAN DARI METADATA ──
        # Kita cek judul, h1, NO PERKARA, dan KLASIFIKASI di metadata
        judul_page   = driver.title.upper()
        h1_text      = ''
        try: h1_text = driver.find_element(By.TAG_NAME, 'h1').text.upper()
        except: pass
        
        no_perkara   = meta.get('no_perkara', '').upper()
        klasifikasi  = meta.get('klasifikasi', '').upper()
        sub_klasifik = meta.get('sub_klasifikasi', '').upper()
        amar_putusan = meta.get('amar_putusan', '').upper()
        
        kata_larangan = ['PENETAPAN', 'AKTA PERDAMAIAN', 'PERDAMAIAN', 'VAN DADING', 'AKTA']
        
        # Cek apakah ada kata larangan di seluruh elemen penting
        if (any(k in judul_page for k in kata_larangan) or 
            any(k in h1_text for k in kata_larangan) or 
            any(k in no_perkara for k in kata_larangan) or 
            any(k in klasifikasi for k in kata_larangan) or 
            any(k in sub_klasifik for k in kata_larangan) or
            'KABUL CABUT' in amar_putusan or 'MENGABULKAN PERMOHONAN Pencabutan' in amar_putusan):
            
            print(f'   ⏭️  [SKIP-BUKAN-PUTUSAN] Terdeteksi sebagai Penetapan/Akta Perdamaian/Pencabutan. Dilewati.')
            gagal_list.append({'url': url_bersih, 'alasan': 'Bukan Putusan Murni (Penetapan/Akta/Cabut)'})
            log_lines.append(f'SKIP [bukan-putusan-murni] {url_bersih}')
            print('-' * 50)
            continue
            
        # ── FILTER KETAT 2: HARUS GUGATAN PDT.G ──────────────────────────────
        if 'PDT.G' not in no_perkara:
            print(f'   ⏭️  [SKIP-BUKAN-PDTG] Bukan gugatan Pdt.G (Mungkin Pdt.P/Penetapan): {no_perkara}. Dilewati.')
            gagal_list.append({'url': url_bersih, 'alasan': f'Bukan Pdt.G: {no_perkara}'})
            log_lines.append(f'SKIP [bukan-pdt-g] {no_perkara} — {url_bersih}')
            print('-' * 50)
            continue
        
        
        # 4. Cari tombol download PDF
        tombol_pdf = None
        taktik_xpath = [
            "//a[contains(@href, 'download_file') and contains(@href, '/pdf/')]",
            "//a[contains(@href, '/pdf/')]",
            "//a[contains(@href, 'download_file')]",
        ]
        for xpt in taktik_xpath:
            try:
                el = driver.find_element(By.XPATH, xpt)
                if el:
                    tombol_pdf = el
                    break
            except:
                continue

        if not tombol_pdf:
            print('   ⏭️  [SKIP] Tombol download PDF tidak ditemukan di halaman ini. Mencoba link berikutnya...')
            gagal_list.append({'url': url_bersih, 'alasan': 'Tombol PDF tidak ada'})
            log_lines.append(f'SKIP [no-btn] {url_bersih}')
            print('-' * 50)
            continue

        url_download_asli = tombol_pdf.get_attribute('href')
        print(f'   🎯 Tombol PDF ditemukan: ...{str(url_download_asli)[-50:]}')

        # 5. Hitung file sebelum klik
        jumlah_sebelum = len([f for f in os.listdir(DIR_PDF) if f.endswith('.pdf')])

        # 6. Klik tombol download
        print('   📥 Mengeklik tombol download...')
        driver.execute_script("arguments[0].click();", tombol_pdf)
        time.sleep(10)  # Ditambah ke 10 detik agar file unduhan biner selesai ditulis ke disk sepenuhnya

        # 7. Deteksi file PDF baru yang masuk
        semua_pdf_baru = sorted([
            f for f in os.listdir(DIR_PDF)
            if f.endswith('.pdf') and not f.endswith('.crdownload')
        ])
        jumlah_sesudah = len(semua_pdf_baru)

        download_sukses = False
        if jumlah_sesudah > jumlah_sebelum:
            file_terbaru = sorted(
                [os.path.join(DIR_PDF, f) for f in semua_pdf_baru],
                key=os.path.getmtime
            )[-1]

            if os.path.abspath(file_terbaru) != path_pdf:
                # SOLUSI UTAMA: Menggunakan os.replace() untuk mem-bypass batasan Windows WinError 183
                os.replace(file_terbaru, path_pdf)

            print(f'   ✅ PDF masuk → {path_pdf}')
            dokumen_terproses += 1
            download_sukses = True

        else:
            # Fallback: cari di folder Downloads pengguna umum
            downloads_dir = os.path.expanduser('~/Downloads')
            pdf_downloads = sorted(
                glob.glob(os.path.join(downloads_dir, '*.pdf')),
                key=os.path.getmtime
            )
            if pdf_downloads:
                file_terbaru = pdf_downloads[-1]
                mtime = os.path.getmtime(file_terbaru)
                if (time.time() - mtime) < 30:  # Baru masuk dalam 30 detik terakhir
                    shutil.move(file_terbaru, path_pdf)
                    print(f'   ✅ PDF dipindah dari Downloads → {path_pdf}')
                    dokumen_terproses += 1
                    download_sukses = True

        if download_sukses:
            log_lines.append(f'OK [{case_id}] {meta["no_perkara"]} — Berhasil')
            data_metadata.append({
                'case_id'      : case_id,
                'no_perkara'   : meta['no_perkara'],
                'tanggal'      : meta['tanggal'],
                'jenis_perkara': meta['jenis_perkara'],
                'pengadilan'   : meta['pengadilan'],
                'pihak'        : meta['pihak'],
                'pasal'        : meta['pasal'],
                'amar_putusan' : meta['amar_putusan'],
                'url_sumber'   : url_bersih,
                'path_pdf'     : path_pdf
            })
        else:
            print('   ⚠️  Gagal mendeteksi masuknya file biner PDF baru. Skip.')
            gagal_list.append({'url': url_bersih, 'alasan': 'PDF tidak terdeteksi'})
            log_lines.append(f'GAGAL [no-file] {url_bersih}')

    except Exception as e:
        print(f'   ❌ Error Operasi Sistem: {e}')
        gagal_list.append({'url': url_bersih, 'alasan': str(e)})
        log_lines.append(f'ERROR {url_bersih}: {e}')

    print('-' * 50)

    if dokumen_terproses % CONFIG['long_pause_every'] == 0 and dokumen_terproses > 0:
        long_pause()
    else:
        human_delay()

berhasil_download = dokumen_terproses
print('=' * 62)
print(f'🎉 TAHAP UNDUH SELESAI! Berkas PDF Berhasil Masuk: {berhasil_download} | Gagal/Skip: {len(gagal_list)}')
print('=' * 62)


⚙️  Folder download Chrome diarahkan ke:
    👉 c:\Users\LENOVO\Documents\Penalaran Komputer\Tugas_3\Tugas3.1\data\raw\pdf

   PROSES UTAMA: PDF DOWNLOADER MURNI (FIXED WINDOWS WINERROR 183)
   Total link: 208 | Target: 80 dokumen

📦 [0/80] Link 1/208
   URL: https://putusan3.mahkamahagung.go.id/direktori/putusan/zaf0e7fc74efbb048336323335393532.html
   ⏭️  [SKIP] Tombol download PDF tidak ditemukan di halaman ini. Mencoba link berikutnya...
--------------------------------------------------
📦 [0/80] Link 2/208
   URL: https://putusan3.mahkamahagung.go.id/direktori/putusan/zaf0e7fbcdf703fcb3a4323335353132.html
   ⏭️  [SKIP] Tombol download PDF tidak ditemukan di halaman ini. Mencoba link berikutnya...
--------------------------------------------------
📦 [0/80] Link 3/208
   URL: https://putusan3.mahkamahagung.go.id/direktori/putusan/zaf0e7fbcdf6662cb3a4323335353132.html
   ⏭️  [SKIP] Tombol download PDF tidak ditemukan di halaman ini. Mencoba link berikutnya...
-------------------------

## Cell 8 — Konversi PDF → TXT

> Jalankan setelah Cell 7 selesai.
> Mengkonversi semua PDF di `data/raw/pdf/` menjadi file teks di `data/raw/txt/`.
> Menggunakan `pdfminer` untuk ekstraksi teks dari PDF.

In [4]:
import os
from pdfminer.high_level import extract_text as pdf_extract

# ─── KONVERSI SEMUA PDF → TXT ─────────────────────────────────────────────────
# Dijalankan setelah Cell 7 selesai (semua PDF sudah terkumpul di data/raw/pdf/)
# Output: data/raw/txt/case_NNN.txt

DIR_PDF = CONFIG['dir_raw_pdf']   # data/raw/pdf/
DIR_TXT = CONFIG['dir_raw_txt']   # data/raw/txt/
os.makedirs(DIR_TXT, exist_ok=True)

pdf_files = sorted([f for f in os.listdir(DIR_PDF) if f.endswith('.pdf')])

print('=' * 62)
print(f'  KONVERSI PDF → TXT')
print(f'  Total PDF ditemukan: {len(pdf_files)}')
print('=' * 62)
print()

konversi_ok  = 0
konversi_err = 0

for fname in pdf_files:
    case_id  = fname.replace('.pdf', '')
    path_pdf = os.path.join(DIR_PDF, fname)
    path_txt = os.path.join(DIR_TXT, f'{case_id}.txt')

    try:
        teks = pdf_extract(path_pdf)
        if teks and len(teks.strip()) > 100:
            with open(path_txt, 'w', encoding='utf-8') as f:
                f.write(teks)
            jml_kata = len(teks.split())
            print(f'  ✅ {case_id}.pdf → {case_id}.txt ({jml_kata:,} kata)')
            konversi_ok += 1
        else:
            print(f'  ⚠️  {case_id}.pdf — teks terlalu pendek, dilewati')
            konversi_err += 1
    except Exception as e:
        print(f'  ❌ {case_id}.pdf — error: {e}')
        konversi_err += 1

print()
print('=' * 62)
print(f'  ✅ Berhasil dikonversi : {konversi_ok} file')
print(f'  ❌ Gagal/dilewati      : {konversi_err} file')
print(f'  📁 Output              : {DIR_TXT}/')
print('=' * 62)


  KONVERSI PDF → TXT
  Total PDF ditemukan: 80

  ✅ case_001.pdf → case_001.txt (21,750 kata)
  ✅ case_002.pdf → case_002.txt (12,799 kata)
  ✅ case_003.pdf → case_003.txt (8,417 kata)
  ✅ case_004.pdf → case_004.txt (2,420 kata)
  ✅ case_005.pdf → case_005.txt (54,474 kata)
  ✅ case_006.pdf → case_006.txt (3,936 kata)
  ✅ case_007.pdf → case_007.txt (20,008 kata)
  ✅ case_008.pdf → case_008.txt (11,746 kata)
  ✅ case_009.pdf → case_009.txt (1,028 kata)
  ✅ case_010.pdf → case_010.txt (32,191 kata)
  ✅ case_011.pdf → case_011.txt (23,178 kata)
  ✅ case_012.pdf → case_012.txt (10,457 kata)
  ✅ case_013.pdf → case_013.txt (12,629 kata)
  ✅ case_014.pdf → case_014.txt (14,671 kata)
  ✅ case_015.pdf → case_015.txt (22,206 kata)
  ✅ case_016.pdf → case_016.txt (6,454 kata)
  ✅ case_017.pdf → case_017.txt (24,148 kata)
  ✅ case_018.pdf → case_018.txt (15,499 kata)
  ✅ case_019.pdf → case_019.txt (13,611 kata)
  ✅ case_020.pdf → case_020.txt (1,310 kata)
  ✅ case_021.pdf → case_021.txt (966 k

## Cell 9 — Preprocessing Teks (Cleaning, Normalisasi, Tokenisasi)

> Jalankan setelah Cell 8 selesai.

| Langkah | Yang Dihapus | Yang Dipertahankan |
|---|---|---|
| Hapus header/footer | Watermark MA RI, Disclaimer, nomor halaman | Seluruh isi putusan |
| Normalisasi spasi | Spasi ganda, tab, baris kosong berlebih | Struktur paragraf |
| Hapus karakter kontrol | Karakter non-printable PDF | Huruf, angka, tanda baca |
| Tokenisasi | — | Angka nominal, pasal, nama pihak |
| Stopwords ringan | Kata sambung umum saja | Semua istilah hukum |

In [19]:
import os, re, json
import pandas as pd

# ── Coba import NLTK (opsional, fallback whitespace tokenizer tersedia) ────────
try:
    from nltk.tokenize import word_tokenize
    import nltk
    nltk.download('punkt', quiet=True)
    nltk.download('punkt_tab', quiet=True)
    USE_NLTK = True
    print("✅ NLTK tersedia — menggunakan word_tokenize")
except ImportError:
    USE_NLTK = False
    print("⚠️  NLTK tidak tersedia — menggunakan whitespace tokenizer")

# ─── STOPWORDS SANGAT RINGAN ──────────────────────────────────────────────────
# HANYA kata sambung & partikel yang 100% tidak bermakna hukum.
# Istilah hukum seperti mengadili, menghukum, pasal TIDAK dimasukkan.
STOPWORDS_RINGAN = {
    'dan', 'di', 'ke', 'dari', 'yang', 'ini', 'itu', 'atau',
    'pada', 'dengan', 'adalah', 'juga', 'oleh', 'serta', 'pula', 'pun', 'nya', 'si',
}

def gabung_huruf_berspasi(teks: str) -> str:
    """
    Gabungkan heading kapital yang huruf-hurufnya dipisah spasi tunggal,
    pola umum di judul resmi putusan PN, contoh:
      'M E N G A D I L I'        -> 'MENGADILI'
      'D E M I  K E A D I L A N' -> 'DEMI KEADILAN' (antar kata tetap terpisah)
    Pola: minimal 3 huruf kapital tunggal berturut-turut dipisah 1 spasi.
    """
    pola = re.compile(r'\b(?:[A-Z]\s){2,}[A-Z]\b')
    return pola.sub(lambda m: m.group().replace(' ', ''), teks)

def hapus_header_footer(teks: str) -> str:
    # Struktur PDF MA RI: konten ada dalam satu baris panjang per halaman,
    # header/footer muncul sebagai baris pendek terpisah.
    # Yang dihapus:
    # - 'Direktori Putusan Mahkamah Agung Republik Indonesia'
    # - 'putusan.mahkamahagung.go.id'
    # - 'Mahkamah Agung Republik Indonesia' (watermark berulang)
    # - 'Halaman X' (nomor halaman footer)
    # - Blok Disclaimer + info kontak Kepaniteraan
    # - Prefix 'Halaman X dari Y Putusan Nomor...' di awal baris konten

    # 1. Ganti form-feed (page break PDF) dengan newline biasa
    teks = teks.replace('\f', '\n')

    # 2. Hapus blok Disclaimer lengkap (multi-baris)
    teks = re.sub(
        r'Disclaimer\s*\nKepaniteraan[\s\S]*?Halaman\s+\d+\s*\n?',
        '\n', teks, flags=re.IGNORECASE
    )

    # 3. Hapus baris header/footer yang berdiri sendiri
    pola_baris_noise = [
        r'^Direktori Putusan Mahkamah Agung Republik Indonesia\s*$',
        r'^putusan\.mahkamahagung\.go\.id\s*$',
        r'^Mahkamah Agung Republik Indonesia\s*$',
        r'^Halaman \d+\s*$',
    ]
    for pola in pola_baris_noise:
        teks = re.sub(pola, '', teks, flags=re.MULTILINE | re.IGNORECASE)

    # 4. Hapus prefix 'Halaman X dari Y Putusan Nomor...' di awal baris konten
    #    PERTAHANKAN sisa konten setelah prefix
    def hapus_prefix_halaman(m):
        sisa = m.group(1)
        return ('\n' + sisa) if sisa.strip() else ''

    teks = re.sub(
        r'Halaman\s+\d+\s+dari\s+\d+\s+Putusan\s+Nomor[:\s]*[\w/\.\(\)]+\s*(.*)',
        hapus_prefix_halaman, teks, flags=re.IGNORECASE
    )
    return teks


def normalisasi(teks: str) -> str:
    # Normalisasi minimal yang AMAN untuk teks hukum.
    # TIDAK dilakukan: lowercase, hapus angka, hapus tanda baca, hapus angka romawi.

    # Hapus karakter kontrol non-printable dari PDF
    teks = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]', '', teks)
    
    # 2. Paksa menjadi lower-case (Diminta di spesifikasi tugas)
    teks = teks.lower()
    
    # 3. Hapus tanda baca / remove punctuation (Diminta di spesifikasi tugas)
    # Melindungi spasi dan alphanumeric, mengubah tanda baca menjadi spasi kosong
    teks = re.sub(r'[^\w\s]', ' ', teks)
    
    # Normalisasi spasi ganda dan tab
    teks = re.sub(r'[ \t]+', ' ', teks)
    
    # Normalisasi baris kosong berlebih (3+) menjadi 2 baris kosong
    teks = re.sub(r'\n{3,}', '\n\n', teks)
    
    # Hapus spasi di awal/akhir setiap baris
    teks = '\n'.join(line.strip() for line in teks.splitlines())
    return teks.strip()


def tokenisasi(teks: str) -> list:
    # Tokenisasi whitespace: aman, mempertahankan angka, pasal, nama pihak, angka romawi
    tokens = word_tokenize(teks) if USE_NLTK else teks.split()
    return [t for t in tokens if t.lower() not in STOPWORDS_RINGAN and len(t) >= 2]


# ─── PROSES SEMUA FILE TXT ────────────────────────────────────────────────────
dir_txt    = CONFIG['dir_raw_txt']
dir_tokens = 'data/processed/tokens'
os.makedirs(dir_tokens, exist_ok=True)

txt_files = sorted([f for f in os.listdir(dir_txt) if f.endswith('.txt')])
hasil     = []

# Kata kunci konten wajib yang harus ada setelah preprocessing
CEKLIS = {
    'menimbang'  : ['menimbang'],
    'wanprestasi': ['wanprestasi', 'wan prestasi', 'cidera janji', 'cedera janji', 'ingkar janji'],
    'penggugat'  : ['penggugat'],
    'tergugat'   : ['tergugat'],
    'mengadili'  : ['mengadili', 'mengabulkan', 'menolak gugatan'],  # variasi kata kunci amar
}


print('=' * 62)
print(f'  PREPROCESSING TEKS — {len(txt_files)} FILE')
print(f'  Stopwords ringan : {len(STOPWORDS_RINGAN)} kata')
print('=' * 62)
print()

for fname in txt_files:
    case_id  = fname.replace('.txt', '')
    path_txt = os.path.join(dir_txt, fname)

    with open(path_txt, 'r', encoding='utf-8') as f:
        teks_raw = f.read()

    kata_sebelum = len(teks_raw.split())
    
    teks_bersih = gabung_huruf_berspasi(teks_raw)       

    # Pipeline preprocessing
    teks_bersih = hapus_header_footer(teks_bersih)
    teks_bersih = normalisasi(teks_bersih)
    
    # Tambah 1 baris setelah normalisasi:
    teks_bersih = re.sub(r'halaman \d+ putusan perdata gugatan nomor[\w\s\/\.]+?pn sby', '', teks_bersih)
    
    tokens      = tokenisasi(teks_bersih)

    kata_sesudah = len(teks_bersih.split())
    reduksi      = round((1 - kata_sesudah / kata_sebelum) * 100, 1) if kata_sebelum > 0 else 0

    # Validasi konten penting masih ada
    hilang = [ 
        label for label, variasi in CEKLIS.items()
        if not any(v in teks_bersih for v in variasi)
    ]
    konten_ok = len(hilang) == 0
    status    = '✅' if konten_ok else f'⚠️  hilang: {hilang}'

    # Timpa file txt dengan versi bersih
    with open(path_txt, 'w', encoding='utf-8') as f:
        f.write(teks_bersih)

    # Simpan tokens ke JSON
    with open(os.path.join(dir_tokens, f'{case_id}_tokens.json'), 'w', encoding='utf-8') as f:
        json.dump({
            'case_id'      : case_id,
            'jumlah_token' : len(tokens),
            'tokens'       : tokens
        }, f, ensure_ascii=False, indent=2)

    hasil.append({
        'case_id'      : case_id,
        'kata_sebelum' : kata_sebelum,
        'kata_sesudah' : kata_sesudah,
        'jumlah_token' : len(tokens),
        'reduksi_pct'  : reduksi,
        'konten_ok'    : konten_ok,
    })

    print(f'  {status}')
    print(f'  {case_id} | {kata_sebelum:,} -> {kata_sesudah:,} kata (reduksi {reduksi}%) | token: {len(tokens):,}')
    print()

# Simpan ringkasan
df = pd.DataFrame(hasil)
df.to_csv('logs/preprocessing_summary.csv', index=False)

print('=' * 62)
print(f'  Selesai: {len(txt_files)} file diproses')
print(f'  Rata-rata reduksi : {df["reduksi_pct"].mean():.1f}%')
print(f'  Konten OK         : {df["konten_ok"].sum()}/{len(df)}')
print(f'  Teks bersih       : {dir_txt}/')
print(f'  Token JSON        : {dir_tokens}/')
print(f'  Ringkasan         : logs/preprocessing_summary.csv')
print('=' * 62)


✅ NLTK tersedia — menggunakan word_tokenize
  PREPROCESSING TEKS — 80 FILE
  Stopwords ringan : 18 kata

  ✅
  case_001 | 16,920 -> 16,920 kata (reduksi 0.0%) | token: 14,570

  ✅
  case_002 | 9,165 -> 9,165 kata (reduksi 0.0%) | token: 7,880

  ✅
  case_003 | 6,515 -> 6,515 kata (reduksi 0.0%) | token: 5,595

  ✅
  case_004 | 1,751 -> 1,751 kata (reduksi 0.0%) | token: 1,442

  ✅
  case_005 | 41,500 -> 41,500 kata (reduksi 0.0%) | token: 35,532

  ✅
  case_006 | 2,930 -> 2,930 kata (reduksi 0.0%) | token: 2,428

  ✅
  case_007 | 14,836 -> 14,836 kata (reduksi 0.0%) | token: 12,719

  ✅
  case_008 | 8,781 -> 8,781 kata (reduksi 0.0%) | token: 7,442

  ⚠️  hilang: ['wanprestasi']
  case_009 | 741 -> 741 kata (reduksi 0.0%) | token: 586

  ✅
  case_010 | 24,445 -> 24,445 kata (reduksi 0.0%) | token: 20,819

  ✅
  case_011 | 16,793 -> 16,793 kata (reduksi 0.0%) | token: 14,180

  ✅
  case_012 | 7,803 -> 7,803 kata (reduksi 0.0%) | token: 6,820

  ✅
  case_013 | 9,543 -> 9,543 kata (reduks

## Cell 10 — Simpan Log & Metadata CSV

In [9]:
import pandas as pd
from datetime import datetime

# ── Simpan log unduhan PDF ────────────────────────────────────────────────
log_lines.append('-' * 50)
log_lines.append(f'Selesai Tahap Unduh: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
log_lines.append(f'PDF Sukses Terunduh: {berhasil_download} | Gagal/Skip: {len(gagal_list)}')

os.makedirs('logs', exist_ok=True)
with open('logs/cleaning.log', 'w', encoding='utf-8') as f:
    f.write('\n'.join(log_lines))
print('✅ logs/cleaning.log (Riwayat Unduhan) tersimpan')

# ── Simpan metadata CSV Sementara ──────────────────────────────────────────
if data_metadata:
    df = pd.DataFrame(data_metadata)
    os.makedirs('data/processed', exist_ok=True)
    df.to_csv('data/processed/metadata_raw.csv', index=False, encoding='utf-8-sig')
    print(f'✅ data/processed/metadata_raw.csv ({len(df)} baris data berkas berhasil diamankan)')
    print()
    
    # Menampilkan kolom metadata yang tersedia murni dari halaman HTML
    cols_tampil = ['case_id', 'no_perkara', 'tanggal', 'pengadilan']
    print(df[cols_tampil].to_string(index=False))
else:
    print('⚠️  Tidak ada metadata halaman web yang bisa disimpan.')

# ── Simpan daftar gagal ────────────────────────────────────────────────────
if gagal_list:
    pd.DataFrame(gagal_list).to_csv('logs/gagal.csv', index=False)
    print(f'\n⚠️  {len(gagal_list)} kasus terlewati/gagal → detail dicatat di logs/gagal.csv')

NameError: name 'log_lines' is not defined

Cell 10 tersebut hanya bisa dijalankan setelah Cell 7 selesai eksekusi di sesi yang sama.
Tapi kalau PDF-nya sudah terunduh semua (80 file ) dan kamu hanya ingin generate log-nya saja tanpa download ulang, pakai cell pengganti ini:

In [10]:
import os, json
import pandas as pd
from datetime import datetime

# ── Reconstruct variabel dari file yang sudah ada ─────────────────────────
DIR_PDF = CONFIG['dir_raw_pdf']

pdf_files = [f for f in os.listdir(DIR_PDF) if f.endswith('.pdf')]
berhasil_download = len(pdf_files)

# Coba load gagal_list dari file jika ada, otherwise kosong
gagal_path = 'logs/gagal.csv'
if os.path.exists(gagal_path):
    df_gagal = pd.read_csv(gagal_path)
    gagal_list = df_gagal.to_dict('records')
else:
    gagal_list = []

# Reconstruct log_lines dari kondisi folder saat ini
log_lines = []
log_lines.append('=' * 50)
log_lines.append('LOG UNDUHAN PDF — REKONSTRUKSI')
log_lines.append(f'Pengadilan : {CONFIG["pengadilan_nama"]}')
log_lines.append(f'Kategori   : {CONFIG["kategori"]}')
log_lines.append(f'Tahun      : {CONFIG["tahun"]}')
log_lines.append('-' * 50)
for fname in sorted(pdf_files):
    fpath = os.path.join(DIR_PDF, fname)
    ukuran = os.path.getsize(fpath) / 1024
    log_lines.append(f'[OK] {fname} ({ukuran:.1f} KB)')
log_lines.append('-' * 50)
log_lines.append(f'Selesai Tahap Unduh: {datetime.now().strftime("%Y-%m-%d %H:%M:%S")}')
log_lines.append(f'PDF Sukses Terunduh: {berhasil_download} | Gagal/Skip: {len(gagal_list)}')
log_lines.append('=' * 50)

# ── Simpan ke cleaning.log ─────────────────────────────────────────────────
os.makedirs('logs', exist_ok=True)
log_path = os.path.join(CONFIG['dir_logs'], 'cleaning.log')
with open(log_path, 'w', encoding='utf-8') as f:
    f.write('\n'.join(log_lines))

print(f'Log disimpan ke {log_path}')
print(f'   PDF tercatat: {berhasil_download} | Gagal: {len(gagal_list)}')

Log disimpan ke logs\cleaning.log
   PDF tercatat: 80 | Gagal: 77


## Cell 11 — Validasi Hasil

In [11]:
import os
import pandas as pd

print('=' * 62)
print('   VALIDASI KUANTITAS UNDUHAN PDF (TAHAP 1.i CBR)')
print('=' * 62)

# Membaca berkas biner asli yang ada di folder
pdf_files = [f for f in os.listdir(CONFIG['dir_raw_pdf']) if f.endswith('.pdf')]

total_pdf = len(pdf_files)
print(f' Lokasi Penyimpanan : {CONFIG["dir_raw_pdf"]}')
print(f' Total PDF Terunduh  : {total_pdf} file')
print()

# ── Evaluasi Syarat Kuantitas Dokumen Sesuai Ketentuan Tugas ──
target_tugas = CONFIG['target_dokumen'] # 50
status_kuantitas = False

if total_pdf >= target_tugas:
    print(f' [LULUS TARGET] Dokumen sangat cukup: {total_pdf} ≥ {target_tugas} (Target Utama Terpenuhi!) ✅')
    status_kuantitas = True
elif total_pdf >= 30:
    print(f' [MEMENUHI MINIMUM] Dokumen aman untuk CBR: {total_pdf} ≥ 30 (Lolos batas minimum regulasi tugas, belum capai target {target_tugas})')
    status_kuantitas = True
else:
    print(f' [BELUM LULUS] Dokumen kurang: {total_pdf} < 30 (Jumlah berkas di bawah standar minimum pengerjaan CBR)')

# ── Catat rincian berkas biner ke log validasi awal ──
detail_pdf = []
for fname in sorted(pdf_files):
    path_file = os.path.join(CONFIG['dir_raw_pdf'], fname)
    ukuran_kb = os.path.getsize(path_file) / 1024 # Konversi bytes ke KiloBytes
    
    detail_pdf.append({
        'Nama File': fname,
        'Ukuran Berkas': f"{ukuran_kb:.2f} KB",
        'Status Download': 'Sukses Murni' if ukuran_kb > 10 else 'Korup/Kosong ⚠️'
    })

df_pdf_val = pd.DataFrame(detail_pdf)
print()
print(df_pdf_val.to_string(index=False))

# Simpan logs checkpoint unduhan
os.makedirs('logs', exist_ok=True)
df_pdf_val.to_csv('logs/validasi_unduhan_pdf.csv', index=False)
print(f'\nRiwayat audit berkas disimpan ke logs/validasi_unduhan_pdf.csv')

   VALIDASI KUANTITAS UNDUHAN PDF (TAHAP 1.i CBR)
 Lokasi Penyimpanan : data/raw/pdf
 Total PDF Terunduh  : 80 file

 [LULUS TARGET] Dokumen sangat cukup: 80 ≥ 80 (Target Utama Terpenuhi!) ✅

   Nama File Ukuran Berkas Status Download
case_001.pdf     451.38 KB    Sukses Murni
case_002.pdf     247.65 KB    Sukses Murni
case_003.pdf     197.63 KB    Sukses Murni
case_004.pdf      97.51 KB    Sukses Murni
case_005.pdf     806.90 KB    Sukses Murni
case_006.pdf     245.18 KB    Sukses Murni
case_007.pdf     338.88 KB    Sukses Murni
case_008.pdf     239.03 KB    Sukses Murni
case_009.pdf      28.74 KB    Sukses Murni
case_010.pdf     468.50 KB    Sukses Murni
case_011.pdf     467.40 KB    Sukses Murni
case_012.pdf     189.22 KB    Sukses Murni
case_013.pdf     220.89 KB    Sukses Murni
case_014.pdf     336.64 KB    Sukses Murni
case_015.pdf     377.59 KB    Sukses Murni
case_016.pdf     131.13 KB    Sukses Murni
case_017.pdf     387.47 KB    Sukses Murni
case_018.pdf     355.48 KB    Suks

## Cell 12 — Ringkasan Akhir Tahap 1

In [12]:
import os

print()
print('============================================================')
print('       RINGKASAN AKHIR TAHAP 1.a: AKUISISI CASE BASE        ')
print('============================================================')
print(f' Pengadilan : PN Surabaya')
print(f' Kategori   : Perdata Gugatan (Wanprestasi)')
print(f' Tahun      : {CONFIG["tahun"]}')
print(f' Kasus di MA: 199 Perkara Terdaftar')
print(f' Target     : {CONFIG["target_dokumen"]} Berkas PDF')
print('=' * 60)

# Menghitung berkas fisik nyata yang ada di folder saat ini
pdf_files = [f for f in os.listdir(CONFIG['dir_raw_pdf']) if f.endswith('.pdf')]
total_pdf = len(pdf_files)

print(f' PDF Berhasil Masuk : {total_pdf} file (-> {CONFIG["dir_raw_pdf"]})')
print(f' Gagal/Terlewati     : {len(gagal_list)} link')

# Kalkulasi pemenuhan kuota target tugas
persentase_target = (total_pdf / CONFIG["target_dokumen"]) * 100
print(f' Progress Kuota     : {persentase_target:.1f}% dari target utama')
print('=' * 60)

print('Output Fisik Tahap Ini Teramankan:')
print(f'   [OK] {CONFIG["dir_raw_pdf"]}*.pdf   - Dokumen hukum biner asli dari MA')
print('   [OK] data/processed/metadata_raw.csv    - Ekstraksi informasi awal web MA')
print('   [OK] logs/cleaning.log                  - Log riwayat transaksi download')
print('   [OK] logs/gagal.csv                     - Daftar link yang bermasalah/no-pdf')
print('-' * 60)

if total_pdf >= 30:
    print(' STATUS: BERKAS AMAN! Silakan lanjut ke CELL TAHAP PERBAIKAN MASSAL')
    print(' (Ekstraksi Teks Mentah, Pembersihan Terstruktur, & Log Keutuhan Data)')
else:
    print(' STATUS: KUOTA BELUM CUKUP! Periksa logs/gagal.csv or jalankan ulang')
    print(' crawler untuk mencukupi batas minimum tugas (minimal 30 dokumen).')
print('=' * 60)


       RINGKASAN AKHIR TAHAP 1.a: AKUISISI CASE BASE        
 Pengadilan : PN Surabaya
 Kategori   : Perdata Gugatan (Wanprestasi)
 Tahun      : 2025
 Kasus di MA: 199 Perkara Terdaftar
 Target     : 80 Berkas PDF
 PDF Berhasil Masuk : 80 file (-> data/raw/pdf)
 Gagal/Terlewati     : 77 link
 Progress Kuota     : 100.0% dari target utama
Output Fisik Tahap Ini Teramankan:
   [OK] data/raw/pdf*.pdf   - Dokumen hukum biner asli dari MA
   [OK] data/processed/metadata_raw.csv    - Ekstraksi informasi awal web MA
   [OK] logs/cleaning.log                  - Log riwayat transaksi download
   [OK] logs/gagal.csv                     - Daftar link yang bermasalah/no-pdf
------------------------------------------------------------
 STATUS: BERKAS AMAN! Silakan lanjut ke CELL TAHAP PERBAIKAN MASSAL
 (Ekstraksi Teks Mentah, Pembersihan Terstruktur, & Log Keutuhan Data)


In [13]:
# ═══════════════════════════════════════════════════════════════════════
# Diagnosa: kenapa kata 'wanprestasi' / 'mengadili' / 'menimbang' hilang
# Jalankan di notebook kamu (folder data/raw/pdf/ sudah ada)
# ═══════════════════════════════════════════════════════════════════════
from pdfminer.high_level import extract_text
import re

CASE_CEK = 'case_009'  # ganti dengan case_id yang mau dicek

path_pdf = f'data/raw/pdf/{CASE_CEK}.pdf'
teks_raw = extract_text(path_pdf)

print(f'Total kata di PDF asli: {len(teks_raw.split())}')
print()

# 1. Cek apakah kata "wanprestasi" (atau variasi penulisan) ADA di PDF asli
for variasi in ['wanprestasi', 'Wanprestasi', 'WANPRESTASI']:
    jumlah = teks_raw.count(variasi)
    print(f'  "{variasi}" muncul {jumlah}x di PDF asli')

print()
# 2. Cek apakah ada hyphen-wrap di sekitar kata itu (kata terpotong garis baru)
pola_hyphen = re.compile(r'\w*-\n\w*')
matches = pola_hyphen.findall(teks_raw)
print(f'Total kata ter-hyphen-wrap: {len(matches)}')
relevan = [m for m in matches if 'pres' in m.lower() or 'wan' in m.lower() or 'tasi' in m.lower()]
print(f'  Yang relevan dengan "wanprestasi": {relevan}')

print()
# 3. Cek apakah ada SPASI ANEH di tengah kata (justified text letter-spacing)
pola_spasi_aneh = re.compile(r'w\s*a\s*n\s*p\s*r\s*e\s*s\s*t\s*a\s*s\s*i', re.IGNORECASE)
match_spasi = pola_spasi_aneh.search(teks_raw)
if match_spasi:
    print(f'  DITEMUKAN pola huruf terpisah spasi: {repr(match_spasi.group())}')
else:
    print('  Tidak ada pola huruf terpisah spasi')

print()
# 4. Cek juga kata sinonim — mungkin dokumen ini pakai istilah lain
for sinonim in ['cidera janji', 'ingkar janji', 'wan prestasi']:
    if sinonim in teks_raw.lower():
        print(f'  ⚠️  Ditemukan SINONIM: "{sinonim}" — mungkin dokumen ini memang tidak pakai kata "wanprestasi" secara literal')

print()
print('--- Cuplikan 300 karakter pertama dari bagian DUDUK PERKARA (untuk cek manual) ---')
idx = teks_raw.lower().find('duduk perkara')
if idx > 0:
    print(teks_raw[idx:idx+400])


Total kata di PDF asli: 1028

  "wanprestasi" muncul 0x di PDF asli
  "Wanprestasi" muncul 0x di PDF asli
  "WANPRESTASI" muncul 0x di PDF asli

Total kata ter-hyphen-wrap: 0
  Yang relevan dengan "wanprestasi": []

  Tidak ada pola huruf terpisah spasi


--- Cuplikan 300 karakter pertama dari bagian DUDUK PERKARA (untuk cek manual) ---


In [14]:
CASE_CEK = 'case_040'
for variasi in ['mengadili', 'Mengadili', 'MENGADILI']:
    jumlah = teks_raw.count(variasi)
    print(f'  "{variasi}" muncul {jumlah}x di PDF asli')

  "mengadili" muncul 2x di PDF asli
  "Mengadili" muncul 0x di PDF asli
  "MENGADILI" muncul 0x di PDF asli


In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# Diagnosa lanjutan: lacak di step mana kata "mengadili" hilang
# ═══════════════════════════════════════════════════════════════════════
from pdfminer.high_level import extract_text
import re

CASE_CEK = 'case_040'
path_pdf = f'data/raw/pdf/{CASE_CEK}.pdf'
teks_raw = extract_text(path_pdf)

# 1. Tampilkan konteks di sekitar SETIAP kemunculan "mengadili" di teks asli
print('=== Konteks di teks RAW (sebelum cleaning apapun) ===')
for m in re.finditer(r'mengadili', teks_raw, re.IGNORECASE):
    start = max(0, m.start() - 150)
    end = min(len(teks_raw), m.end() + 150)
    print(f'--- posisi {m.start()} ---')
    print(repr(teks_raw[start:end]))
    print()
    



# 2. Jalankan fungsi cleaning yang ada di notebook kamu (copy persis dari Cell 9)
def hapus_header_footer(teks):
    teks = teks.replace('\f', '\n')
    teks = re.sub(r'Disclaimer\s*\nKepaniteraan[\s\S]*?Halaman\s+\d+\s*\n?', '\n', teks, flags=re.IGNORECASE)
    for pola in [r'^Direktori Putusan Mahkamah Agung Republik Indonesia\s*$', r'^putusan\.mahkamahagung\.go\.id\s*$',
                 r'^Mahkamah Agung Republik Indonesia\s*$', r'^Halaman \d+\s*$']:
        teks = re.sub(pola, '', teks, flags=re.MULTILINE | re.IGNORECASE)
    def hapus_prefix_halaman(m):
        sisa = m.group(1)
        return ('\n' + sisa) if sisa.strip() else ''
    teks = re.sub(r'Halaman\s+\d+\s+dari\s+\d+\s+Putusan\s+Nomor[:\s]*[\w/\.\(\)]+\s*(.*)', hapus_prefix_halaman, teks, flags=re.IGNORECASE)
    return teks

def normalisasi(teks):
    teks = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]', '', teks)
    teks = teks.lower()
    teks = re.sub(r'[^\w\s]', ' ', teks)
    teks = re.sub(r'[ \t]+', ' ', teks)
    teks = re.sub(r'\n{3,}', '\n\n', teks)
    teks = '\n'.join(line.strip() for line in teks.splitlines())
    return teks.strip()

t1 = hapus_header_footer(teks_raw)
print('=== Setelah hapus_header_footer: "mengadili" masih ada?', 'mengadili' in t1.lower(), '===')
if 'mengadili' not in t1.lower():
    print('  >>> HILANG DI STEP INI (hapus_header_footer)')

t2 = normalisasi(t1)
t3 = re.sub(r'halaman \d+ putusan perdata gugatan nomor[\w\s\/\.]+?pn sby', '', t2)
print('=== Setelah normalisasi + regex terakhir: "mengadili" masih ada?', 'mengadili' in t3, '===')
if 'mengadili' not in t3 and 'mengadili' in t1.lower():
    print('  >>> HILANG DI STEP INI (normalisasi atau regex terakhir)')


=== Konteks di teks RAW (sebelum cleaning apapun) ===
=== Setelah hapus_header_footer: "mengadili" masih ada? False ===
  >>> HILANG DI STEP INI (hapus_header_footer)
=== Setelah normalisasi + regex terakhir: "mengadili" masih ada? False ===


In [16]:
# ═══════════════════════════════════════════════════════════════════════
# Diagnosa v2: cek konsistensi & cari karakter tersembunyi di sekitar "mengadili"
# ═══════════════════════════════════════════════════════════════════════
from pdfminer.high_level import extract_text
import re

CASE_CEK = 'case_040'
path_pdf = f'data/raw/pdf/{CASE_CEK}.pdf'
teks_raw = extract_text(path_pdf)

print('Total karakter:', len(teks_raw))
print('Jumlah "mengadili" via str.count():', teks_raw.count('mengadili'))
print()

# Cari pakai str.find() murni (tanpa regex sama sekali)
posisi = []
start = 0
while True:
    idx = teks_raw.find('mengadili', start)
    if idx == -1:
        break
    posisi.append(idx)
    start = idx + 1

print(f'Posisi ditemukan via str.find(): {posisi}')
print()

for idx in posisi:
    konteks = teks_raw[max(0,idx-80):idx+90]
    print(f'--- posisi {idx} ---')
    print('Teks biasa :', konteks)
    print('Representasi byte (cari karakter aneh):')
    print(' ', [c if c.isprintable() else f'[U+{ord(c):04X}]' for c in konteks])
    print()

# Cross-check: apakah regex literal r'mengadili' (TANPA flag apapun) ketemu?
print('Cek regex polos tanpa IGNORECASE:', len(re.findall(r'mengadili', teks_raw)), 'match')
print('Cek regex dengan IGNORECASE:', len(re.findall(r'mengadili', teks_raw, re.IGNORECASE)), 'match')
print('Cek regex dengan re.UNICODE eksplisit:', len(re.findall(r'mengadili', teks_raw, re.IGNORECASE | re.UNICODE)), 'match')


Total karakter: 92873
Jumlah "mengadili" via str.count(): 0

Posisi ditemukan via str.find(): []

Cek regex polos tanpa IGNORECASE: 0 match
Cek regex dengan IGNORECASE: 0 match
Cek regex dengan re.UNICODE eksplisit: 0 match


In [17]:
# ═══════════════════════════════════════════════════════════════════════
# Cek identitas dokumen: putusan APA sebenarnya yang ada di case_040.pdf
# untuk pastikan apakah ini dokumen yang sama dengan sebelumnya atau bukan
# ═══════════════════════════════════════════════════════════════════════
from pdfminer.high_level import extract_text
import re

CASE_CEK = 'case_040'
path_pdf = f'data/raw/pdf/{CASE_CEK}.pdf'
teks_raw = extract_text(path_pdf)

# Ambil Nomor Perkara dari teks
m = re.search(r'Nomor\s+(\d+/Pdt\.G/\d+/PN\s*\w*)', teks_raw, re.IGNORECASE)
print('Nomor Perkara terdeteksi:', m.group(1) if m else 'TIDAK TERDETEKSI')

# Cek apakah ada di metadata_raw.csv (mapping case_id -> url_sumber & no_perkara dari hasil scraping)
import pandas as pd
try:
    df_meta = pd.read_csv('data/processed/metadata_raw.csv')
    row = df_meta[df_meta['case_id'] == CASE_CEK]
    if not row.empty:
        print()
        print('Menurut metadata_raw.csv (hasil scraping):')
        print('  no_perkara :', row.iloc[0]['no_perkara'])
        print('  url_sumber :', row.iloc[0]['url_sumber'])
    else:
        print(f'⚠️  {CASE_CEK} tidak ditemukan di metadata_raw.csv')
except FileNotFoundError:
    print('⚠️  metadata_raw.csv tidak ditemukan')

# Cek timestamp file PDF — kapan terakhir diubah/dibuat
import os
mtime = os.path.getmtime(path_pdf)
from datetime import datetime
print()
print('PDF terakhir dimodifikasi:', datetime.fromtimestamp(mtime).strftime('%Y-%m-%d %H:%M:%S'))

print()
print('--- 300 karakter pertama dokumen (untuk identifikasi manual) ---')
print(teks_raw[:300])


Nomor Perkara terdeteksi: 126/Pdt.G/2025/PN SbyDEMI

Menurut metadata_raw.csv (hasil scraping):
  no_perkara : 126/Pdt.G/2025/PN Sby
  url_sumber : https://putusan3.mahkamahagung.go.id/direktori/putusan/zaf0888c7fc2d6da9818313330393036.html

PDF terakhir dimodifikasi: 2026-06-15 21:20:39

--- 300 karakter pertama dokumen (untuk identifikasi manual) ---
Direktori Putusan Mahkamah Agung Republik Indonesia
putusan.mahkamahagung.go.id

Mahkamah Agung Republik Indonesia
Mahkamah Agung Republik Indonesia
Mahkamah Agung Republik Indonesia
Mahkamah Agung Republik Indonesia
Mahkamah Agung Republik Indonesia

Disclaimer
Kepaniteraan Mahkamah Agung Republik 


In [18]:
# ═══════════════════════════════════════════════════════════════════════
# Cek apakah "MENGADILI" ditulis dengan huruf berspasi (M E N G A D I L I)
# gaya umum di judul/kop resmi dokumen pengadilan Indonesia
# ═══════════════════════════════════════════════════════════════════════
from pdfminer.high_level import extract_text
import re

CASE_CEK = 'case_040'
path_pdf = f'data/raw/pdf/{CASE_CEK}.pdf'
teks_raw = extract_text(path_pdf)

# Pola: tiap huruf M-E-N-G-A-D-I-L-I dipisah spasi (boleh 0-2 spasi antar huruf)
pola_spasi = re.compile(r'M\s{0,2}E\s{0,2}N\s{0,2}G\s{0,2}A\s{0,2}D\s{0,2}I\s{0,2}L\s{0,2}I', re.IGNORECASE)
matches = list(pola_spasi.finditer(teks_raw))
print(f'Pola huruf berspasi "M E N G A D I L I" ditemukan: {len(matches)}x')
for m in matches:
    print(f'  posisi {m.start()}: {repr(teks_raw[max(0,m.start()-30):m.end()+30])}')

print()
# Cek juga DEMI KEADILAN sebagai pembanding (biasanya juga style yang sama di kop dokumen)
pola_demi = re.compile(r'D\s{0,2}E\s{0,2}M\s{0,2}I', re.IGNORECASE)
m2 = pola_demi.search(teks_raw)
print('Contoh "DEMI" (untuk verifikasi gaya tipografi serupa):', repr(m2.group()) if m2 else 'tidak ditemukan')

print()
# Tampilkan juga sekitar kata "Pengadilan Negeri ... yang memeriksa dan" untuk lihat kalimat boilerplate awal
idx_periksa = teks_raw.lower().find('memeriksa dan')
if idx_periksa > 0:
    print('--- Konteks "memeriksa dan ___" (boilerplate awal) ---')
    print(repr(teks_raw[idx_periksa:idx_periksa+150]))


Pola huruf berspasi "M E N G A D I L I" ditemukan: 1x
  posisi 90082: '4 3348 (ext.318)\n\nHalaman 28\n\nM E N G A D I L I DALAM EKSEPSI :-Mengabulkan E'

Contoh "DEMI" (untuk verifikasi gaya tipografi serupa): 'DEMI'

--- Konteks "memeriksa dan ___" (boilerplate awal) ---
'memeriksa dan memutus perkaraperdata pada tingkat pertama, telah menjatuhkan putusan sebagai berikut dalamperkara gugatan antara:  PT. Puncak Kertajay'


In [20]:
# ═══════════════════════════════════════════════════════════════════════
# Cek BATCH: 18 case yang masih hilang 'wanprestasi' setelah fix sinonim
# Tujuan: cari tahu istilah apa yang sebenarnya dipakai di tiap dokumen,
# supaya bisa diputuskan: tambah ke daftar sinonim, atau keluarkan dari case base
# ═══════════════════════════════════════════════════════════════════════
from pdfminer.high_level import extract_text
import re, os

CASE_BERMASALAH = [
    'case_009', 'case_020', 'case_021', 'case_022', 'case_028',
    'case_038', 'case_039', 'case_042', 'case_045', 'case_048',
    'case_051', 'case_056', 'case_057', 'case_059', 'case_062',
    'case_065', 'case_067', 'case_079',
]

# Kandidat istilah pengganti yang mungkin dipakai dokumen wanprestasi
KANDIDAT_ISTILAH = [
    'wanprestasi', 'wan prestasi', 'cidera janji', 'cedera janji',
    'ingkar janji', 'wanprestesi', 'wan-prestasi',
]

print(f'{"case_id":<12} | {"Nomor Perkara":<28} | Istilah ditemukan')
print('-' * 80)

hasil_ringkasan = []

for case_id in CASE_BERMASALAH:
    path_pdf = f'data/raw/pdf/{case_id}.pdf'
    if not os.path.exists(path_pdf):
        print(f'{case_id:<12} | FILE TIDAK DITEMUKAN')
        continue

    teks_raw = extract_text(path_pdf)
    teks_lower = teks_raw.lower()

    no_perkara_m = re.search(r'Nomor\s+(\d+/Pdt\.G/\d+/PN\s*\w*)', teks_raw, re.IGNORECASE)
    no_perkara = no_perkara_m.group(1) if no_perkara_m else '???'

    ditemukan = [istilah for istilah in KANDIDAT_ISTILAH if istilah in teks_lower]

    status = ', '.join(ditemukan) if ditemukan else '❌ TIDAK ADA istilah apapun'
    print(f'{case_id:<12} | {no_perkara:<28} | {status}')

    hasil_ringkasan.append({
        'case_id': case_id,
        'no_perkara': no_perkara,
        'istilah_ditemukan': ditemukan,
        'perlu_dicek_manual': len(ditemukan) == 0,
    })

print()
print('=' * 80)
butuh_cek_manual = [h['case_id'] for h in hasil_ringkasan if h['perlu_dicek_manual']]
print(f'Case yang BENAR-BENAR tidak ada istilah wanprestasi/sinonim apapun: {len(butuh_cek_manual)}')
print(f'  {butuh_cek_manual}')
print()
print('Untuk case-case ini, tampilkan cuplikan "TENTANG DUDUK PERKARA" agar bisa dicek manual relevansinya:')
print()

for case_id in butuh_cek_manual:
    path_pdf = f'data/raw/pdf/{case_id}.pdf'
    teks_raw = extract_text(path_pdf)
    idx = teks_raw.upper().find('TENTANG DUDUK PERKARA')
    print(f'--- {case_id} ---')
    if idx > 0:
        print(teks_raw[idx:idx+400].strip())
    else:
        print('(bagian "TENTANG DUDUK PERKARA" tidak ditemukan, tampilkan 400 char pertama)')
        print(teks_raw[:400].strip())
    print()


case_id      | Nomor Perkara                | Istilah ditemukan
--------------------------------------------------------------------------------
case_009     | 587/Pdt.G/2025/PN Sby        | ❌ TIDAK ADA istilah apapun
case_020     | 651/Pdt.G/2025/PN SbyPada    | ❌ TIDAK ADA istilah apapun
case_021     | 782/Pdt.G/2025/PN Sby        | ❌ TIDAK ADA istilah apapun
case_022     | 939/Pdt.G/2025/PN Sby        | ❌ TIDAK ADA istilah apapun
case_028     | 908/Pdt.G/2025/PN Sby        | ❌ TIDAK ADA istilah apapun
case_038     | 991/Pdt.G/2025/PN SbyDEMI    | ❌ TIDAK ADA istilah apapun
case_039     | 125/Pdt.G/2025/PN SbyPada    | ❌ TIDAK ADA istilah apapun
case_042     | 785/Pdt.G/2025/PN            | ❌ TIDAK ADA istilah apapun
case_045     | 733/PDT.G/2025/PN SBY        | ❌ TIDAK ADA istilah apapun
case_048     | 467/Pdt.G/2025/PN            | ❌ TIDAK ADA istilah apapun
case_051     | 484/Pdt.G/2025/PN Sby        | ❌ TIDAK ADA istilah apapun
case_056     | 668/Pdt.G/2025/PN            | ❌ TIDA

In [21]:
# ═══════════════════════════════════════════════════════════════════════
# Cek batch v2: lewati boilerplate, cari isi ASLI dokumen + heading alternatif
# ═══════════════════════════════════════════════════════════════════════
from pdfminer.high_level import extract_text
import re, os

CASE_BERMASALAH = [
    'case_009', 'case_020', 'case_021', 'case_022', 'case_028',
    'case_038', 'case_039', 'case_042', 'case_045', 'case_048',
    'case_051', 'case_056', 'case_057', 'case_059', 'case_062',
    'case_065', 'case_067', 'case_079',
]

# Heading/kata kunci yang menandakan jenis & isi putusan
HEADING_ALTERNATIF = [
    'tentang duduk perkara', 'dalam pokok perkara', 'tentang pertimbangan hukum',
    'verstek', 'tidak hadir', 'dengan tidak hadirnya', 'mengadili dengan verstek',
    'gugatan sederhana', 'gugatan default',
]

for case_id in CASE_BERMASALAH:
    path_pdf = f'data/raw/pdf/{case_id}.pdf'
    if not os.path.exists(path_pdf):
        continue
    teks_raw = extract_text(path_pdf)
    teks_lower = teks_raw.lower()

    # Lewati boilerplate: cari posisi setelah blok Disclaimer KEDUA (karena
    # blok pertama ada di awal dokumen, isi sebenarnya biasanya mulai
    # setelah "PUTUSAN" / "Nomor .../Pdt.G/..." muncul untuk kedua kalinya)
    posisi_nomor = [m.start() for m in re.finditer(r'P\s*U\s*T\s*U\s*S\s*A\s*N', teks_raw, re.IGNORECASE)]

    print(f'=== {case_id} ===')
    print(f'  Total kata: {len(teks_raw.split())}')
    print(f'  Heading "PUTUSAN" ditemukan di posisi: {posisi_nomor[:3]}')

    heading_ada = [h for h in HEADING_ALTERNATIF if h in teks_lower]
    print(f'  Heading/kata kunci alternatif ditemukan: {heading_ada if heading_ada else "TIDAK ADA"}')

    # Tampilkan isi mulai dari kemunculan KEDUA "PUTUSAN" (lewati kop/disclaimer awal)
    if len(posisi_nomor) >= 2:
        mulai = posisi_nomor[1]
        print(f'  --- Isi (mulai dari kemunculan ke-2 "PUTUSAN", {600} karakter) ---')
        print(' ', teks_raw[mulai:mulai+600].replace('\n', ' | '))
    else:
        print('  --- Isi (600 karakter pertama, karena heading PUTUSAN cuma 1x atau 0x) ---')
        print(' ', teks_raw[:600].replace('\n', ' | '))
    print()


=== case_009 ===
  Total kata: 1028
  Heading "PUTUSAN" ditemukan di posisi: [10, 52, 2978]
  Heading/kata kunci alternatif ditemukan: ['tidak hadir']
  --- Isi (mulai dari kemunculan ke-2 "PUTUSAN", 600 karakter) ---
  putusan.mahkamahagung.go.id |  | Mahkamah Agung Republik Indonesia | Mahkamah Agung Republik Indonesia | Mahkamah Agung Republik Indonesia | Mahkamah Agung Republik Indonesia | Mahkamah Agung Republik Indonesia |  | Disclaimer | Kepaniteraan Mahkamah Agung Republik Indonesia berusaha untuk selalu mencantumkan informasi paling kini dan akurat sebagai bentuk komitmen Mahkamah Agung untuk pelayanan publik, transparansi dan akuntabilitas | pelaksanaan fungsi peradilan. Namun dalam hal-hal tertentu masih dimungkinkan terjadi permasalahan teknis terkait dengan akurasi dan keterkinian informasi yang kami sajikan, hal ma

=== case_020 ===
  Total kata: 1310
  Heading "PUTUSAN" ditemukan di posisi: [10, 52, 2533]
  Heading/kata kunci alternatif ditemukan: TIDAK ADA
  --- Isi (mu

In [22]:
# ═══════════════════════════════════════════════════════════════════════
# Cek batch v3: ambil isi dari kemunculan KE-3 "PUTUSAN" (bukan ke-2)
# ═══════════════════════════════════════════════════════════════════════
from pdfminer.high_level import extract_text
import re, os

CASE_BERMASALAH = [
    'case_009', 'case_020', 'case_021', 'case_022', 'case_028',
    'case_038', 'case_039', 'case_042', 'case_045', 'case_048',
    'case_051', 'case_056', 'case_057', 'case_059', 'case_062',
    'case_065', 'case_067', 'case_079',
]

HEADING_ALTERNATIF = [
    'tentang duduk perkara', 'dalam pokok perkara', 'tentang pertimbangan hukum',
    'verstek', 'tidak hadir', 'dengan tidak hadirnya', 'mengadili dengan verstek',
    'gugatan sederhana', 'gugatan default', 'wanprestasi', 'cidera janji', 'ingkar janji',
]

for case_id in CASE_BERMASALAH:
    path_pdf = f'data/raw/pdf/{case_id}.pdf'
    if not os.path.exists(path_pdf):
        continue
    teks_raw = extract_text(path_pdf)
    teks_lower = teks_raw.lower()

    posisi_nomor = [m.start() for m in re.finditer(r'P\s*U\s*T\s*U\s*S\s*A\s*N', teks_raw, re.IGNORECASE)]

    print(f'=== {case_id} ({len(teks_raw.split())} kata) ===')
    heading_ada = [h for h in HEADING_ALTERNATIF if h in teks_lower]
    print(f'  Kata kunci ditemukan: {heading_ada if heading_ada else "TIDAK ADA SAMA SEKALI"}')

    # Ambil dari kemunculan KE-3 (index 2), karena index 0 & 1 selalu boilerplate
    if len(posisi_nomor) >= 3:
        mulai = posisi_nomor[2]
        print('  --- Isi sebenarnya (800 karakter) ---')
        print(' ', teks_raw[mulai:mulai+800].replace('\n', ' | '))
    else:
        print('  (Heading PUTUSAN kurang dari 3x, tampilkan 800 char terakhir sebagai gantinya)')
        print(' ', teks_raw[-800:].replace('\n', ' | '))
    print()


=== case_009 (1028 kata) ===
  Kata kunci ditemukan: ['tidak hadir']
  --- Isi sebenarnya (800 karakter) ---
  Putusan Mahkamah Agung Republik Indonesia | putusan.mahkamahagung.go.id |  | Mahkamah Agung Republik Indonesia | Mahkamah Agung Republik Indonesia | Mahkamah Agung Republik Indonesia | Mahkamah Agung Republik Indonesia | Mahkamah Agung Republik Indonesia |  | Disclaimer | Kepaniteraan Mahkamah Agung Republik Indonesia berusaha untuk selalu mencantumkan informasi paling kini dan akurat sebagai bentuk komitmen Mahkamah Agung untuk pelayanan publik, transparansi dan akuntabilitas | pelaksanaan fungsi peradilan. Namun dalam hal-hal tertentu masih dimungkinkan terjadi permasalahan teknis terkait dengan akurasi dan keterkinian informasi yang kami sajikan, hal mana akan terus kami perbaiki dari waktu kewaktu. | Dalam hal Anda menemukan inakurasi informasi yang termuat pada situs ini atau informasi yang seharusnya ada, n

=== case_020 (1310 kata) ===
  Kata kunci ditemukan: TIDAK ADA 

In [2]:
# ═══════════════════════════════════════════════════════════════════════
# Klasifikasi final 18 case: BUANG (Penetapan/Akta) vs PERTAHANKAN (Putusan asli)
# ═══════════════════════════════════════════════════════════════════════
from pdfminer.high_level import extract_text
import re, os

CASE_BERMASALAH = [
    'case_009', 'case_020', 'case_021', 'case_022', 'case_028',
    'case_038', 'case_039', 'case_042', 'case_045', 'case_048',
    'case_051', 'case_056', 'case_057', 'case_059', 'case_062',
    'case_065', 'case_067', 'case_079',
]

def gabung_huruf_berspasi(teks):
    pola = re.compile(r'\b(?:[A-Z]\s){2,}[A-Z]\b')
    return pola.sub(lambda m: m.group().replace(' ', ''), teks)

hasil = []

for case_id in CASE_BERMASALAH:
    path_pdf = f'data/raw/pdf/{case_id}.pdf'
    if not os.path.exists(path_pdf):
        continue
    teks_raw = extract_text(path_pdf)

    # PENTING: gabung huruf berspasi DULU sebelum cek jenis dokumen,
    # supaya "P E N E T A P A N" ikut terdeteksi sebagai "PENETAPAN"
    teks_gabung = gabung_huruf_berspasi(teks_raw)

    # Ambil 2000 karakter pertama SETELAH boilerplate (lewati 2x "PUTUSAN" awal)
    posisi = [m.start() for m in re.finditer(r'PUTUSAN', teks_gabung, re.IGNORECASE)]
    mulai = posisi[2] if len(posisi) >= 3 else 0
    konten_awal = teks_gabung[mulai:mulai+2000]

    is_penetapan = bool(re.search(r'\bPENETAPAN\b', konten_awal, re.IGNORECASE))
    is_akta = bool(re.search(r'AKTA PERDAMAIAN', konten_awal, re.IGNORECASE))

    if is_penetapan or is_akta:
        kategori = 'BUANG — ' + ('PENETAPAN' if is_penetapan else 'AKTA PERDAMAIAN')
    else:
        kategori = 'PUTUSAN ASLI (perlu cek istilah wanprestasi manual)'

    no_perkara_m = re.search(r'Nomor\s+(\d+/Pdt\.G/\d+/PN[\.\s]*\w*)', teks_raw, re.IGNORECASE)
    no_perkara = no_perkara_m.group(1) if no_perkara_m else '???'

    hasil.append((case_id, no_perkara, kategori))
    print(f'{case_id:<12} | {no_perkara:<28} | {kategori}')

print()
buang = [h[0] for h in hasil if 'BUANG' in h[2]]
pertahankan = [h[0] for h in hasil if 'BUANG' not in h[2]]
print('=' * 80)
print(f'Total HARUS DIBUANG (Penetapan/Akta Perdamaian): {len(buang)}')
print(f'  {buang}')
print(f'Total kemungkinan PUTUSAN ASLI (perlu cek istilah): {len(pertahankan)}')
print(f'  {pertahankan}')


case_009     | 587/Pdt.G/2025/PN Sby        | BUANG — PENETAPAN
case_020     | 651/Pdt.G/2025/PN SbyPada    | PUTUSAN ASLI (perlu cek istilah wanprestasi manual)
case_021     | 782/Pdt.G/2025/PN Sby        | BUANG — PENETAPAN
case_022     | 939/Pdt.G/2025/PN Sby        | BUANG — PENETAPAN
case_028     | 908/Pdt.G/2025/PN Sby        | BUANG — PENETAPAN
case_038     | 991/Pdt.G/2025/PN SbyDEMI    | PUTUSAN ASLI (perlu cek istilah wanprestasi manual)
case_039     | 125/Pdt.G/2025/PN SbyPada    | BUANG — AKTA PERDAMAIAN
case_042     | 785/Pdt.G/2025/PN.Sby        | PUTUSAN ASLI (perlu cek istilah wanprestasi manual)
case_045     | 733/PDT.G/2025/PN SBY        | BUANG — AKTA PERDAMAIAN
case_048     | 467/Pdt.G/2025/PN.Sby        | BUANG — PENETAPAN
case_051     | 484/Pdt.G/2025/PN Sby        | BUANG — PENETAPAN
case_056     | 668/Pdt.G/2025/PN.SbyDEMI    | PUTUSAN ASLI (perlu cek istilah wanprestasi manual)
case_057     | 756/Pdt.G/2025/PN Sby        | BUANG — PENETAPAN
case_059     | 416/P

In [3]:
# ═══════════════════════════════════════════════════════════════════════
# Scan SELURUH 80 case: deteksi Penetapan/Akta Perdamaian yang lolos filter scraping
# (termasuk yang sudah ✅ di validasi CEKLIS sebelumnya, karena bisa saja
# lolos cuma karena kata "wanprestasi" disebut di dalil, padahal dokumennya
# sendiri PENETAPAN/AKTA PERDAMAIAN, bukan putusan akhir)
# ═══════════════════════════════════════════════════════════════════════
from pdfminer.high_level import extract_text
import re, os

def gabung_huruf_berspasi(teks):
    pola = re.compile(r'\b(?:[A-Z]\s){2,}[A-Z]\b')
    return pola.sub(lambda m: m.group().replace(' ', ''), teks)

DIR_PDF = 'data/raw/pdf'
semua_case = sorted([f.replace('.pdf','') for f in os.listdir(DIR_PDF) if f.endswith('.pdf')])

hasil = []

for case_id in semua_case:
    path_pdf = os.path.join(DIR_PDF, f'{case_id}.pdf')
    teks_raw = extract_text(path_pdf)
    teks_gabung = gabung_huruf_berspasi(teks_raw)

    # Ambil bagian konten asli (lewati boilerplate awal, ambil dari kemunculan ke-3 "PUTUSAN")
    posisi = [m.start() for m in re.finditer(r'PUTUSAN', teks_gabung, re.IGNORECASE)]
    mulai = posisi[2] if len(posisi) >= 3 else 0
    konten = teks_gabung[mulai:mulai+2000]

    is_penetapan = bool(re.search(r'\bPENETAPAN\b', konten, re.IGNORECASE))
    is_akta      = bool(re.search(r'AKTA\s+PERDAMAIAN', konten, re.IGNORECASE))
    is_pencabutan = bool(re.search(r'PENCABUTAN', konten, re.IGNORECASE))

    if is_penetapan or is_akta or is_pencabutan:
        jenis = []
        if is_penetapan: jenis.append('PENETAPAN')
        if is_akta: jenis.append('AKTA PERDAMAIAN')
        if is_pencabutan: jenis.append('PENCABUTAN')
        hasil.append((case_id, 'BUANG', '+'.join(jenis)))
    else:
        hasil.append((case_id, 'OK', '-'))

print(f'{"case_id":<12} | {"Status":<8} | Jenis Masalah')
print('-' * 50)
bermasalah = []
for case_id, status, jenis in hasil:
    if status == 'BUANG':
        bermasalah.append(case_id)
        print(f'{case_id:<12} | {status:<8} | {jenis}')

print()
print('=' * 50)
print(f'Total case yang TERNYATA Penetapan/Akta/Pencabutan: {len(bermasalah)} dari {len(semua_case)}')
print(f'  {bermasalah}')


KeyboardInterrupt: 

In [4]:
# Versi dengan progress bar, supaya kelihatan jalan dan tidak terasa "macet"
from pdfminer.high_level import extract_text
import re, os
from tqdm import tqdm

def gabung_huruf_berspasi(teks):
    pola = re.compile(r'\b(?:[A-Z]\s){2,}[A-Z]\b')
    return pola.sub(lambda m: m.group().replace(' ', ''), teks)

DIR_PDF = 'data/raw/pdf'
semua_case = sorted([f.replace('.pdf','') for f in os.listdir(DIR_PDF) if f.endswith('.pdf')])

hasil = []
bermasalah = []

for case_id in tqdm(semua_case, desc='Scanning PDF'):
    path_pdf = os.path.join(DIR_PDF, f'{case_id}.pdf')
    teks_raw = extract_text(path_pdf)
    teks_gabung = gabung_huruf_berspasi(teks_raw)

    posisi = [m.start() for m in re.finditer(r'PUTUSAN', teks_gabung, re.IGNORECASE)]
    mulai = posisi[2] if len(posisi) >= 3 else 0
    konten = teks_gabung[mulai:mulai+2000]

    is_penetapan = bool(re.search(r'\bPENETAPAN\b', konten, re.IGNORECASE))
    is_akta      = bool(re.search(r'AKTA\s+PERDAMAIAN', konten, re.IGNORECASE))
    is_pencabutan = bool(re.search(r'PENCABUTAN', konten, re.IGNORECASE))

    if is_penetapan or is_akta or is_pencabutan:
        jenis = []
        if is_penetapan: jenis.append('PENETAPAN')
        if is_akta: jenis.append('AKTA PERDAMAIAN')
        if is_pencabutan: jenis.append('PENCABUTAN')
        bermasalah.append((case_id, '+'.join(jenis)))

print()
print(f'Total case bermasalah: {len(bermasalah)} dari {len(semua_case)}')
for c, j in bermasalah:
    print(f'  {c}: {j}')


Scanning PDF: 100%|██████████| 80/80 [03:31<00:00,  2.65s/it]


Total case bermasalah: 21 dari 80
  case_006: PENETAPAN
  case_009: PENETAPAN+PENCABUTAN
  case_021: PENETAPAN
  case_022: PENETAPAN
  case_026: PENETAPAN
  case_028: PENETAPAN
  case_038: PENCABUTAN
  case_039: AKTA PERDAMAIAN
  case_043: PENETAPAN
  case_045: AKTA PERDAMAIAN
  case_047: PENETAPAN
  case_048: PENETAPAN
  case_051: PENETAPAN
  case_056: PENCABUTAN
  case_057: PENETAPAN
  case_059: PENCABUTAN
  case_062: PENETAPAN
  case_065: PENETAPAN+PENCABUTAN
  case_067: PENETAPAN
  case_078: PENCABUTAN
  case_079: PENCABUTAN


In [5]:
# ═══════════════════════════════════════════════════════════════════════
# Scan FULL v2: cek SELURUH isi dokumen (bukan cuma 2000 char awal),
# supaya tidak ada yang lolos seperti case_004 kemarin
# ═══════════════════════════════════════════════════════════════════════
from pdfminer.high_level import extract_text
import re, os
from tqdm import tqdm

def gabung_huruf_berspasi(teks):
    pola = re.compile(r'\b(?:[A-Z]\s){2,}[A-Z]\b')
    return pola.sub(lambda m: m.group().replace(' ', ''), teks)

DIR_PDF = 'data/raw/pdf'
semua_case = sorted([f.replace('.pdf','') for f in os.listdir(DIR_PDF) if f.endswith('.pdf')])

bermasalah = []

for case_id in tqdm(semua_case, desc='Scanning PDF (full text)'):
    path_pdf = os.path.join(DIR_PDF, f'{case_id}.pdf')
    teks_raw = extract_text(path_pdf)
    teks_gabung = gabung_huruf_berspasi(teks_raw)

    # Cek di SELURUH dokumen, fokus ke kemunculan sebagai JUDUL/HEADING
    # (diawali baris baru atau didahului spasi, huruf besar semua, bukan
    # disebut di tengah kalimat dalil seperti "...akta perdamaian yang dibuat...")
    pola_judul_penetapan = re.compile(r'(?:^|\n)\s*PENETAPAN\b', re.IGNORECASE)
    pola_judul_akta       = re.compile(r'(?:^|\n)\s*AKTA\s+PERDAMAIAN\b', re.IGNORECASE)
    pola_judul_pencabutan = re.compile(r'PENCABUTAN\s+PERKARA|PENETAPAN\s+PENCABUTAN', re.IGNORECASE)

    jenis = []
    if pola_judul_penetapan.search(teks_gabung):
        jenis.append('PENETAPAN')
    if pola_judul_akta.search(teks_gabung):
        jenis.append('AKTA PERDAMAIAN')
    if pola_judul_pencabutan.search(teks_gabung):
        jenis.append('PENCABUTAN')

    if jenis:
        bermasalah.append((case_id, '+'.join(jenis)))

print()
print(f'Total case bermasalah (scan full): {len(bermasalah)} dari {len(semua_case)}')
for c, j in bermasalah:
    print(f'  {c}: {j}')

# Cek khusus case_004 untuk verifikasi
print()
print('=== Verifikasi khusus case_004 ===')
path_004 = os.path.join(DIR_PDF, 'case_004.pdf')
if os.path.exists(path_004):
    t = extract_text(path_004)
    tg = gabung_huruf_berspasi(t)
    print('Mengandung "AKTA PERDAMAIAN"?', 'AKTA PERDAMAIAN' in tg.upper())
    idx = tg.upper().find('AKTA PERDAMAIAN')
    if idx > 0:
        print('Konteks:', repr(tg[max(0,idx-50):idx+100]))


Scanning PDF (full text): 100%|██████████| 80/80 [03:31<00:00,  2.65s/it]



Total case bermasalah (scan full): 18 dari 80
  case_006: PENETAPAN
  case_009: PENCABUTAN
  case_019: PENCABUTAN
  case_021: PENCABUTAN
  case_022: PENCABUTAN
  case_026: PENCABUTAN
  case_038: PENCABUTAN
  case_044: PENCABUTAN
  case_045: AKTA PERDAMAIAN
  case_047: PENCABUTAN
  case_051: PENCABUTAN
  case_056: PENETAPAN+PENCABUTAN
  case_057: PENCABUTAN
  case_059: PENCABUTAN
  case_065: PENCABUTAN
  case_067: PENCABUTAN
  case_078: PENETAPAN+PENCABUTAN
  case_079: PENCABUTAN

=== Verifikasi khusus case_004 ===
Mengandung "AKTA PERDAMAIAN"? True
Konteks: 'o.id    Telp : 021-384 3348 (ext.318)\n\nHalaman 1\n\nAKTA PERDAMAIANNomor 578/Pdt.G/2025/PN SbyPada hari Rabu, tanggal 17 September 2025, dalam persidang'


In [6]:
# ═══════════════════════════════════════════════════════════════════════
# Scan FINAL v3: hapus \b di akhir pola (pdfminer sering nempelkan heading
# dengan kata berikutnya tanpa spasi, contoh "PERDAMAIANNomor")
# ═══════════════════════════════════════════════════════════════════════
from pdfminer.high_level import extract_text
import re, os
from tqdm import tqdm

def gabung_huruf_berspasi(teks):
    pola = re.compile(r'\b(?:[A-Z]\s){2,}[A-Z]\b')
    return pola.sub(lambda m: m.group().replace(' ', ''), teks)

DIR_PDF = 'data/raw/pdf'
semua_case = sorted([f.replace('.pdf','') for f in os.listdir(DIR_PDF) if f.endswith('.pdf')])

bermasalah = []

# PENTING: \b dihapus di AKHIR pola, hanya dipakai di AWAL pola
pola_penetapan  = re.compile(r'(?:^|\n)\s*PENETAPAN', re.IGNORECASE)
pola_akta       = re.compile(r'(?:^|\n)\s*AKTA\s*PERDAMAIAN', re.IGNORECASE)
pola_pencabutan = re.compile(r'PENETAPAN\s*PENCABUTAN|PENCABUTAN\s*PERKARA', re.IGNORECASE)

for case_id in tqdm(semua_case, desc='Scanning PDF (final)'):
    path_pdf = os.path.join(DIR_PDF, f'{case_id}.pdf')
    teks_raw = extract_text(path_pdf)
    teks_gabung = gabung_huruf_berspasi(teks_raw)

    jenis = []
    if pola_penetapan.search(teks_gabung):
        jenis.append('PENETAPAN')
    if pola_akta.search(teks_gabung):
        jenis.append('AKTA PERDAMAIAN')
    if pola_pencabutan.search(teks_gabung):
        jenis.append('PENCABUTAN')

    if jenis:
        bermasalah.append((case_id, '+'.join(jenis)))

print()
print(f'Total case bermasalah (FINAL): {len(bermasalah)} dari {len(semua_case)}')
for c, j in bermasalah:
    print(f'  {c}: {j}')

print()
case_id_bermasalah = [c for c, j in bermasalah]
print('List case_id untuk dikeluarkan dari case base:')
print(case_id_bermasalah)


Scanning PDF (final): 100%|██████████| 80/80 [03:32<00:00,  2.66s/it]


Total case bermasalah (FINAL): 22 dari 80
  case_004: AKTA PERDAMAIAN
  case_006: PENETAPAN
  case_009: PENCABUTAN
  case_019: PENCABUTAN
  case_020: AKTA PERDAMAIAN
  case_021: PENCABUTAN
  case_022: PENCABUTAN
  case_026: PENCABUTAN
  case_038: PENCABUTAN
  case_039: AKTA PERDAMAIAN
  case_044: PENETAPAN+PENCABUTAN
  case_045: AKTA PERDAMAIAN
  case_047: PENCABUTAN
  case_051: PENCABUTAN
  case_056: PENETAPAN+PENCABUTAN
  case_057: PENCABUTAN
  case_059: PENCABUTAN
  case_065: PENCABUTAN
  case_067: PENCABUTAN
  case_073: AKTA PERDAMAIAN
  case_078: PENETAPAN+PENCABUTAN
  case_079: PENCABUTAN

List case_id untuk dikeluarkan dari case base:
['case_004', 'case_006', 'case_009', 'case_019', 'case_020', 'case_021', 'case_022', 'case_026', 'case_038', 'case_039', 'case_044', 'case_045', 'case_047', 'case_051', 'case_056', 'case_057', 'case_059', 'case_065', 'case_067', 'case_073', 'case_078', 'case_079']


# 22 dari 80 harus dibuang, sisa 58 case bersih 

In [7]:
# ═══════════════════════════════════════════════════════════════════════
# Bersihkan case base: hapus 22 case yang ternyata Penetapan/Akta/Pencabutan
# ═══════════════════════════════════════════════════════════════════════
import os, glob, json
import pandas as pd

CASE_DIBUANG = [
    'case_004', 'case_006', 'case_009', 'case_019', 'case_020', 'case_021',
    'case_022', 'case_026', 'case_038', 'case_039', 'case_044', 'case_045',
    'case_047', 'case_051', 'case_056', 'case_057', 'case_059', 'case_065',
    'case_067', 'case_073', 'case_078', 'case_079',
]

print(f'Akan membuang {len(CASE_DIBUANG)} case dari case base...\n')

# 1. Hapus file PDF
for case_id in CASE_DIBUANG:
    path = f'data/raw/pdf/{case_id}.pdf'
    if os.path.exists(path):
        os.remove(path)
        print(f'  🗑️  {path}')

# 2. Hapus file TXT mentah
for case_id in CASE_DIBUANG:
    path = f'data/raw/{case_id}.txt'
    if os.path.exists(path):
        os.remove(path)
        print(f'  🗑️  {path}')

# 3. Hapus file token JSON (hasil Cell 9)
for case_id in CASE_DIBUANG:
    path = f'data/processed/tokens/{case_id}_tokens.json'
    if os.path.exists(path):
        os.remove(path)
        print(f'  🗑️  {path}')

# 4. Bersihkan metadata_raw.csv (hasil Cell 7)
path_meta = 'data/processed/metadata_raw.csv'
if os.path.exists(path_meta):
    df_meta = pd.read_csv(path_meta)
    sebelum = len(df_meta)
    df_meta = df_meta[~df_meta['case_id'].isin(CASE_DIBUANG)]
    df_meta.to_csv(path_meta, index=False, encoding='utf-8-sig')
    print(f'\n✅ metadata_raw.csv dibersihkan: {sebelum} -> {len(df_meta)} baris')

# 5. Bersihkan logs/preprocessing_summary.csv (kalau ada)
path_summary = 'logs/preprocessing_summary.csv'
if os.path.exists(path_summary):
    df_sum = pd.read_csv(path_summary)
    sebelum = len(df_sum)
    df_sum = df_sum[~df_sum['case_id'].isin(CASE_DIBUANG)]
    df_sum.to_csv(path_summary, index=False)
    print(f'✅ preprocessing_summary.csv dibersihkan: {sebelum} -> {len(df_sum)} baris')

# 6. Bersihkan data/processed/cases.csv / cases.json KALAU sudah dibuat di Tahap 2
for ext, loader in [('csv', pd.read_csv), ('json', None)]:
    path = f'data/processed/cases.{ext}'
    if os.path.exists(path):
        if ext == 'csv':
            df = loader(path)
            sebelum = len(df)
            df = df[~df['case_id'].isin(CASE_DIBUANG)]
            df.to_csv(path, index=False, encoding='utf-8-sig')
            print(f'✅ cases.csv dibersihkan: {sebelum} -> {len(df)} baris')
        else:
            with open(path, 'r', encoding='utf-8') as f:
                data = json.load(f)
            sebelum = len(data)
            data = [d for d in data if d.get('case_id') not in CASE_DIBUANG]
            with open(path, 'w', encoding='utf-8') as f:
                json.dump(data, f, ensure_ascii=False, indent=2)
            print(f'✅ cases.json dibersihkan: {sebelum} -> {len(data)} baris')

sisa = len([f for f in os.listdir('data/raw/pdf') if f.endswith('.pdf')]) if os.path.exists('data/raw/pdf') else 0
print(f'\n🎯 Sisa case bersih di case base: {sisa}')


Akan membuang 22 case dari case base...

  🗑️  data/raw/pdf/case_004.pdf
  🗑️  data/raw/pdf/case_006.pdf
  🗑️  data/raw/pdf/case_009.pdf
  🗑️  data/raw/pdf/case_019.pdf
  🗑️  data/raw/pdf/case_020.pdf
  🗑️  data/raw/pdf/case_021.pdf
  🗑️  data/raw/pdf/case_022.pdf
  🗑️  data/raw/pdf/case_026.pdf
  🗑️  data/raw/pdf/case_038.pdf
  🗑️  data/raw/pdf/case_039.pdf
  🗑️  data/raw/pdf/case_044.pdf
  🗑️  data/raw/pdf/case_045.pdf
  🗑️  data/raw/pdf/case_047.pdf
  🗑️  data/raw/pdf/case_051.pdf
  🗑️  data/raw/pdf/case_056.pdf
  🗑️  data/raw/pdf/case_057.pdf
  🗑️  data/raw/pdf/case_059.pdf
  🗑️  data/raw/pdf/case_065.pdf
  🗑️  data/raw/pdf/case_067.pdf
  🗑️  data/raw/pdf/case_073.pdf
  🗑️  data/raw/pdf/case_078.pdf
  🗑️  data/raw/pdf/case_079.pdf
  🗑️  data/raw/case_004.txt
  🗑️  data/raw/case_006.txt
  🗑️  data/raw/case_009.txt
  🗑️  data/raw/case_019.txt
  🗑️  data/raw/case_020.txt
  🗑️  data/raw/case_021.txt
  🗑️  data/raw/case_022.txt
  🗑️  data/raw/case_026.txt
  🗑️  data/raw/case_038.txt
  🗑

## (Opsional) Rename case_id agar berurutan kembali

Jalankan cell ini **setelah** Cell 47 (pembersihan 22 case yang ternyata Penetapan/Akta/Pencabutan) dan **sebelum** memulai Tahap 2.

**Catatan:** Cell ini bersifat opsional — Tahap 2 tetap berjalan normal walau nama file bolong, karena loop-nya membaca  dari isi , bukan dari urutan nomor file. Jalankan cell ini hanya jika ingin struktur folder  terlihat rapi berurutan ( s.d. ).

Cell ini akan mengubah nama secara **konsisten di 4 tempat sekaligus**:
1. File PDF di 
2. File TXT di 
3. File token JSON di  (jika ada)
4. Kolom  di 

Mapping nama lama -> nama baru disimpan ke  sebagai jejak audit.

In [8]:
# ═══════════════════════════════════════════════════════════════════════
# Cell (Opsional) — Rename case_id agar berurutan kembali (case_001..case_N)
# Jalankan SETELAH Cell 47 (pembersihan 22 case) dan SEBELUM mulai Tahap 2.
# Mengubah nama di SEMUA lokasi sekaligus: PDF, TXT, token JSON, metadata_raw.csv
# ═══════════════════════════════════════════════════════════════════════
import os, json
import pandas as pd

# 1. Ambil daftar case_id yang TERSISA, urutkan sesuai urutan numerik lama
path_meta = 'data/processed/metadata_raw.csv'
df_meta = pd.read_csv(path_meta, encoding='utf-8-sig')
df_meta = df_meta.sort_values('case_id').reset_index(drop=True)

# 2. Buat mapping: nama lama -> nama baru yang berurutan
mapping = {}
for idx, old_id in enumerate(df_meta['case_id'], start=1):
    new_id = f'case_{idx:03d}'
    mapping[old_id] = new_id

print(f'Total case yang akan di-rename: {len(mapping)}')
print('Contoh mapping (5 pertama):')
for old, new in list(mapping.items())[:5]:
    print(f'  {old}  ->  {new}')
print('...')
print()

# 3. Rename file PDF
renamed_pdf = 0
for old_id, new_id in mapping.items():
    src = f'data/raw/pdf/{old_id}.pdf'
    dst = f'data/raw/pdf/__tmp__{new_id}.pdf'  # nama sementara, hindari tabrakan
    if os.path.exists(src):
        os.rename(src, dst)
        renamed_pdf += 1

# Hapus prefix __tmp__ setelah semua file lama selesai dipindah (hindari overwrite)
for old_id, new_id in mapping.items():
    tmp = f'data/raw/pdf/__tmp__{new_id}.pdf'
    final = f'data/raw/pdf/{new_id}.pdf'
    if os.path.exists(tmp):
        os.rename(tmp, final)
print(f'✅ PDF di-rename: {renamed_pdf} file')

# 4. Rename file TXT mentah
renamed_txt = 0
for old_id, new_id in mapping.items():
    src = f'data/raw/{old_id}.txt'
    dst = f'data/raw/__tmp__{new_id}.txt'
    if os.path.exists(src):
        os.rename(src, dst)
        renamed_txt += 1
for old_id, new_id in mapping.items():
    tmp = f'data/raw/__tmp__{new_id}.txt'
    final = f'data/raw/{new_id}.txt'
    if os.path.exists(tmp):
        os.rename(tmp, final)
print(f'✅ TXT di-rename: {renamed_txt} file')

# 5. Rename file token JSON (hasil Cell 9 Tahap 1), kalau ada
renamed_tok = 0
dir_tokens = 'data/processed/tokens'
if os.path.exists(dir_tokens):
    for old_id, new_id in mapping.items():
        src = f'{dir_tokens}/{old_id}_tokens.json'
        dst = f'{dir_tokens}/__tmp__{new_id}_tokens.json'
        if os.path.exists(src):
            os.rename(src, dst)
            renamed_tok += 1
    for old_id, new_id in mapping.items():
        tmp = f'{dir_tokens}/__tmp__{new_id}_tokens.json'
        final = f'{dir_tokens}/{new_id}_tokens.json'
        if os.path.exists(tmp):
            os.rename(tmp, final)
print(f'✅ Token JSON di-rename: {renamed_tok} file')

# 6. Update isi metadata_raw.csv — kolom case_id DAN kolom url/path yang mungkin menyebut nama file lama
df_meta['case_id'] = df_meta['case_id'].map(mapping)
df_meta = df_meta.sort_values('case_id').reset_index(drop=True)
df_meta.to_csv(path_meta, index=False, encoding='utf-8-sig')
print(f'✅ metadata_raw.csv diperbarui: case_id sudah berurutan ({df_meta["case_id"].iloc[0]} .. {df_meta["case_id"].iloc[-1]})')

# 7. Update juga preprocessing_summary.csv kalau ada (hasil Cell 9/10 Tahap 1)
path_summary = 'logs/preprocessing_summary.csv'
if os.path.exists(path_summary):
    df_sum = pd.read_csv(path_summary)
    df_sum['case_id'] = df_sum['case_id'].map(mapping)
    df_sum.to_csv(path_summary, index=False)
    print('✅ logs/preprocessing_summary.csv diperbarui')

# 8. Simpan mapping lama->baru sebagai log, untuk jaga-jaga/audit
os.makedirs('logs', exist_ok=True)
with open('logs/rename_mapping.json', 'w', encoding='utf-8') as f:
    json.dump(mapping, f, ensure_ascii=False, indent=2)
print('✅ Mapping lama->baru disimpan ke logs/rename_mapping.json')

print()
print('=' * 62)
print(f'SELESAI. Case base sekarang berurutan: case_001 .. case_{len(mapping):03d}')
print('=' * 62)


Total case yang akan di-rename: 58
Contoh mapping (5 pertama):
  case_001  ->  case_001
  case_002  ->  case_002
  case_003  ->  case_003
  case_005  ->  case_004
  case_007  ->  case_005
...

✅ PDF di-rename: 58 file
✅ TXT di-rename: 58 file
✅ Token JSON di-rename: 58 file
✅ metadata_raw.csv diperbarui: case_id sudah berurutan (case_001 .. case_058)
✅ logs/preprocessing_summary.csv diperbarui
✅ Mapping lama->baru disimpan ke logs/rename_mapping.json

SELESAI. Case base sekarang berurutan: case_001 .. case_058
